# AMML Ranking — финальное решение

## 1. Импорты и настройки

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HOME", "/kaggle/working/hf_cache" if os.path.isdir("/kaggle/working") else "/content/hf_cache")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys
import re
import gc
import json
import time
import html
import math
import hashlib
import zipfile
import unicodedata
import warnings
import platform
import tempfile
import shutil
from pathlib import Path
from importlib.metadata import version as package_version, PackageNotFoundError

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, HashingVectorizer
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 180)

RANDOM_STATE = 42
N_SPLITS = 5

PREFER_FEATURE_CACHE = False
ALLOW_RICH98_REBUILD = True
RUN_JINA = True

RUN_NATASHA = True
RUN_TFIDF_BM25 = True
RUN_WORD_TFIDF = True
RUN_CHAR_HASH = False
RUN_BM25 = True
RUN_BGE_M3 = True
RUN_BGE_RERANKER = True
RUN_QWEN_RERANKER = True
AUTO_INSTALL_OPTIONAL_PACKAGES = False
ALLOW_CPU_NEURAL = False

BGE_M3_MODEL_ID = "BAAI/bge-m3"
BGE_RERANKER_MODEL_ID = "BAAI/bge-reranker-v2-m3"
QWEN_MODEL_ID = "Qwen/Qwen3-Reranker-0.6B"

BGE_MAX_LENGTH = 384
BGE_BATCH_SIZE = 16
RERANKER_MAX_LENGTH = 384
RERANKER_BATCH_SIZE = 16
QWEN_MAX_LENGTH = 512
QWEN_BATCH_SIZE = 4

# Jina v3.5
TOP_K = 30
CANDIDATE_RANK_SIGNALS = [
    "bge_reranker_score",
    "qwen_reranker_score",
    "bge_m3_dense",
    "bge_m3_sparse",
    "bm25_query_document",
    "tfidf_word_query_document",
]
JINA_MODEL_ID = "jinaai/jina-reranker-v3.5"
JINA_MODEL_REVISION = "e8a93f33f0b22108f8c2364f8484ce3422552fbc"
JINA_MAX_DOC_TOKENS_PER_DOC = 512
JINA_TOTAL_DOC_TOKEN_BUDGET = 12288
JINA_MAX_QUERY_TOKENS = 256
JINA_TOKEN_TRUNCATION = "prefix"
JINA_INPUT_NORMALIZATION = "experiment11_conservative_lower_preserve_yo"
JINA_MEMORY_PREFLIGHT = True
JINA_PREFLIGHT_NUM_TASKS = 5
JINA_PREFLIGHT_STOP_AT_GB = 11.0
JINA_CHECKPOINT_EVERY_QUERIES = 25
JINA_MEMORY_CLEANUP_EVERY_QUERIES = 25
JINA_MEMORY_WARN_FREE_GB = 2.0
JINA_MEMORY_CLEANUP_ENABLED = True
JINA_CUDA_IPC_COLLECT = True
JINA_MAX_SESSION_HOURS = None

RANKER_PARAMS = {
    "num_leaves": 19,
    "min_child_samples": 35,
    "reg_lambda": 8.0,
    "learning_rate": 0.035,
}
N_ESTIMATORS = 1500
EARLY_STOPPING_ROUNDS = 100

OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="amml_ranking_"))
JINA_MEMORY_TELEMETRY_FILE = OUTPUT_DIR / "jina_memory_telemetry.csv"


print("System:", platform.system(), "|", platform.platform())
print("Python:", sys.version.split()[0])
print("Jina:", JINA_MODEL_ID, "| revision:", JINA_MODEL_REVISION)
print("Top-K:", TOP_K, "| token limits:", JINA_MAX_DOC_TOKENS_PER_DOC,
      JINA_TOTAL_DOC_TOKEN_BUDGET, JINA_MAX_QUERY_TOKENS)

## 2. Исходные данные

In [ ]:
MANUAL_PATHS = {"train": None, "test": None, "sample_submission": None}
SEARCH_ROOTS = [Path("/content"), Path("/kaggle/input"), Path.cwd(), Path("/mnt/data")]
SEARCH_ROOTS = [p for p in SEARCH_ROOTS if p.exists()]
EXCLUDED = {".cache", "sample_data", "__pycache__", ".ipynb_checkpoints"}

def find_files(filename):
    found = []
    for root in SEARCH_ROOTS:
        try:
            for path in root.rglob(filename):
                if path.is_file() and not any(part in EXCLUDED for part in path.parts):
                    found.append(path.resolve())
        except (PermissionError, OSError):
            pass
    return sorted(set(found), key=lambda p: str(p))

def resolve_input(key, filename):
    manual = MANUAL_PATHS[key]
    if manual:
        path = Path(manual).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    found = find_files(filename)
    if len(found) == 1:
        return found[0]
    if not found:
        raise FileNotFoundError(f"Не найден {filename}. Загрузи три CSV в одну папку или укажи путь в MANUAL_PATHS.")
    parents = {p.parent for p in found}
    if len(parents) == 1:
        return found[0]
    raise FileNotFoundError(f"Найдено несколько {filename}. Укажи точный путь в MANUAL_PATHS: {found[:10]}")

TRAIN_PATH = resolve_input("train", "train.csv")
TEST_PATH = resolve_input("test", "test.csv")
SAMPLE_PATH = resolve_input("sample_submission", "sample_submission.csv")


def read_csv(path):
    errors = []
    for engine in ("c", "python"):
        try:
            kwargs = {"encoding": "utf-8-sig", "engine": engine, "dtype": {"id": "string"}, "on_bad_lines": "error"}
            if engine == "c":
                kwargs["low_memory"] = False
            return pd.read_csv(path, **kwargs)
        except Exception as exc:
            errors.append(f"{engine}: {exc}")
    raise RuntimeError(f"Не удалось прочитать {path}: {'; '.join(errors)}")

train = read_csv(TRAIN_PATH)
test = read_csv(TEST_PATH)
sample_submission = read_csv(SAMPLE_PATH)

print("Train:", train.shape, "| Test:", test.shape)
print("Queries:", train["query_id"].nunique(), "| Test queries:", test["query_id"].nunique())
print("Labels:", train["relevance"].value_counts().sort_index().to_dict())


## 3. Формируем 98 признаков

In [ ]:
CACHE_READY = False
CACHE_SOURCE = "recomputed from train.csv and test.csv"


#### Проверяем схему и пропуски

In [ ]:
if not CACHE_READY:
    COMMON_COLUMNS = [
        "id", "query_id", "document_id", "article_id",
        "query", "title", "text", "passage_index",
        "article_passage_count", "article_char_count", "ingest_source"
    ]

    missing_train = sorted(set(COMMON_COLUMNS + ["relevance"]) - set(train.columns))
    missing_test = sorted(set(COMMON_COLUMNS) - set(test.columns))
    assert not missing_train, f"В train отсутствуют: {missing_train}"
    assert not missing_test, f"В test отсутствуют: {missing_test}"
    assert "relevance" not in test.columns, "В test не должно быть relevance"

    schema_report = pd.DataFrame({
        "column": COMMON_COLUMNS,
        "train_dtype": [str(train[c].dtype) for c in COMMON_COLUMNS],
        "test_dtype": [str(test[c].dtype) for c in COMMON_COLUMNS],
        "train_missing": [int(train[c].isna().sum()) for c in COMMON_COLUMNS],
        "test_missing": [int(test[c].isna().sum()) for c in COMMON_COLUMNS],
    })

    print("Unique IDs:", train.id.nunique(), "/", len(train), "|", test.id.nunique(), "/", len(test))
    print("Duplicate IDs:", train.id.duplicated().sum(), "/", test.id.duplicated().sum())
    print("Duplicate full rows:", train.duplicated().sum(), "/", test.duplicated().sum())
    display(schema_report)

    missing_rows = []
    for split_name, frame in [("train", train), ("test", test)]:
        for col in frame.columns:
            null_count = int(frame[col].isna().sum())
            empty_count = 0
            if frame[col].dtype == "object":
                empty_count = int((frame[col].notna() & frame[col].astype(str).str.strip().eq("")).sum())
            missing_rows.append({
                "split": split_name, "column": col,
                "null_count": null_count, "empty_count": empty_count,
                "null_pct": 100 * null_count / len(frame),
            })
    missing_report = pd.DataFrame(missing_rows)
    display(missing_report.sort_values(["split", "null_pct"], ascending=[True, False]))

#### Приводим числовые метаданные к единому виду

In [ ]:
if not CACHE_READY:
    def parse_locale_number(series):
        s = series.astype("string")
        s = s.str.replace(r"[\s\u00A0\u202F\u2009]+", "", regex=True)
        s = s.str.replace(",", ".", regex=False)
        return pd.to_numeric(s, errors="coerce")


    def numeric_audit(frame, split_name):
        raw = pd.to_numeric(frame["article_char_count"], errors="coerce")
        parsed = parse_locale_number(frame["article_char_count"])
        idx = pd.to_numeric(frame["passage_index"], errors="coerce")
        cnt = pd.to_numeric(frame["article_passage_count"], errors="coerce")
        return {
            "split": split_name,
            "rows": len(frame),
            "article_char_count_original_missing": int(frame.article_char_count.isna().sum()),
            "article_char_count_recovered": int((raw.isna() & parsed.notna()).sum()),
            "passage_index_missing": int(idx.isna().sum()),
            "passage_index_negative": int((idx < 0).sum()),
            "article_passage_count_missing": int(cnt.isna().sum()),
            "article_passage_count_negative": int((cnt < 0).sum()),
            "passage_index_ge_positive_count": int((idx.notna() & cnt.notna() & (cnt > 0) & (idx >= cnt)).sum()),
        }

    numeric_report = pd.DataFrame([numeric_audit(train, "train"), numeric_audit(test, "test")])
    display(numeric_report)

#### Нормализуем текст

In [ ]:
if not CACHE_READY:
    TEXT_COLUMNS = ["query", "title", "text"]
    SPECIAL_CHARS = {
        "NBSP": "\u00A0", "NARROW_NBSP": "\u202F", "THIN_SPACE": "\u2009",
        "ZERO_WIDTH_SPACE": "\u200B", "BOM": "\uFEFF"
    }

    unicode_rows = []
    for split_name, frame in [("train", train), ("test", test)]:
        for col in TEXT_COLUMNS:
            s = frame[col].fillna("").astype(str)
            rec = {"split": split_name, "column": col,
                   "rows_with_combining_marks": int(s.str.contains(r"[\u0300-\u036f]", regex=True).sum())}
            for name, char in SPECIAL_CHARS.items():
                rec[f"rows_with_{name.lower()}"] = int(s.str.contains(re.escape(char), regex=True).sum())
            unicode_rows.append(rec)
    unicode_report = pd.DataFrame(unicode_rows)
    display(unicode_report)


    def normalize_text(value, fold_yo=False):
        if pd.isna(value):
            return ""
        s = html.unescape(str(value))
        s = unicodedata.normalize("NFKC", s)
        s = re.sub(r"[\u0300-\u036f]", "", s)
        for char in ["\u00A0", "\u202F", "\u2009", "\u200A", "\u200B", "\u2060", "\uFEFF"]:
            s = s.replace(char, " ")
        s = re.sub(r"\s+", " ", s).strip().lower()
        if fold_yo:
            s = s.replace("ё", "е")
        return s


    train_fe = train.copy()
    test_fe = test.copy()
    for frame in [train_fe, test_fe]:
        for col in TEXT_COLUMNS:
            frame[f"{col}_norm"] = frame[col].map(normalize_text)
            frame[f"{col}_yo_fold"] = frame[f"{col}_norm"].str.replace("ё", "е", regex=False)
        frame["title_missing_clean"] = frame["title"].isna().astype("int8")

    print("Normalization example:")
    display(train_fe[["query", "query_norm", "query_yo_fold", "title", "title_norm"]].head(5))

#### Формируем признаки метаданных

In [ ]:
if not CACHE_READY:
    def add_metadata_features(frame):
        out = frame.copy()
        idx = pd.to_numeric(out["passage_index"], errors="coerce")
        count_raw = pd.to_numeric(out["article_passage_count"], errors="coerce")
        char_count = parse_locale_number(out["article_char_count"])

        out["passage_index_num"] = idx
        out["passage_index_missing"] = idx.isna().astype("int8")
        out["article_passage_count_num_raw"] = count_raw
        out["article_passage_count_missing"] = count_raw.isna().astype("int8")
        out["article_passage_count_negative"] = (count_raw < 0).fillna(False).astype("int8")
        out["article_passage_count_clean"] = count_raw.where(count_raw > 0)
        out["article_char_count_num"] = char_count.where(char_count >= 0)
        out["article_char_count_missing"] = char_count.isna().astype("int8")
        out["article_char_count_parse_failed"] = (out["article_char_count"].notna() & char_count.isna()).astype("int8")

        valid_pair = idx.notna() & out["article_passage_count_clean"].notna()
        out["passage_index_ge_count"] = (
            valid_pair & (idx >= out["article_passage_count_clean"])
        ).astype("int8")
        out["passage_position_ratio_index_over_count"] = (
            idx / out["article_passage_count_clean"]
        )
        denom_last = out["article_passage_count_clean"] - 1
        out["passage_position_ratio_zero_to_last"] = idx / denom_last.where(denom_last > 0)

        out["article_char_count_per_passage"] = (
            out["article_char_count_num"] / out["article_passage_count_clean"]
        )
        out["text_len_chars"] = out["text_norm"].str.len()
        out["title_len_chars"] = out["title_norm"].str.len()
        out["query_len_chars"] = out["query_norm"].str.len()
        out["text_to_article_char_ratio"] = (
            out["text_len_chars"] / out["article_char_count_num"]
        )
        return out

    train_fe = add_metadata_features(train_fe)
    test_fe = add_metadata_features(test_fe)

    print("Metadata feature shapes:", train_fe.shape, test_fe.shape)
    display(numeric_report)

#### Считаем лексические признаки

In [ ]:
if not CACHE_READY:
    TOKEN_RE = re.compile(r"[0-9a-zа-яё]+", flags=re.IGNORECASE)
    NUMBER_RE = re.compile(r"\d+")
    QUESTION_WORDS = {
        "кто", "что", "где", "куда", "откуда", "когда", "почему", "зачем",
        "сколько", "какой", "какая", "какое", "какие", "чей", "чья", "чье", "чьи",
        "кем", "кому", "чем", "откуда", "впервые", "в каком"
    }


    def tokenize(text):
        return TOKEN_RE.findall(str(text).lower())


    def overlap_features(query, target, prefix):
        q = set(tokenize(query))
        t_list = tokenize(target)
        t = set(t_list)
        overlap = q & t
        return {
            f"{prefix}_overlap": len(overlap),
            f"{prefix}_coverage": len(overlap) / max(len(q), 1),
            f"{prefix}_jaccard": len(overlap) / max(len(q | t), 1),
        }


    def build_lexical_features(frame):
        rows = []
        for q, title, text in zip(frame["query_norm"], frame["title_norm"], frame["text_norm"]):
            row = {}
            row.update(overlap_features(q, title, "query_title"))
            row.update(overlap_features(q, text, "query_text"))

            q_tokens = tokenize(q)
            q_set = set(q_tokens)
            text_tokens = tokenize(text)
            title_tokens = tokenize(title)

            q_numbers = set(NUMBER_RE.findall(q))
            text_numbers = set(NUMBER_RE.findall(text))
            row["query_token_count"] = len(q_tokens)
            row["query_unique_token_count"] = len(q_set)
            row["title_token_count"] = len(title_tokens)
            row["text_token_count"] = len(text_tokens)
            row["query_number_count"] = len(q_numbers)
            row["query_number_overlap"] = len(q_numbers & text_numbers)
            row["query_question_word"] = next((w for w in q_tokens if w in QUESTION_WORDS), "__NONE__")

            first_positions = [text.find(token) for token in q_set if token and text.find(token) >= 0]
            first_pos = min(first_positions) if first_positions else np.nan
            row["first_match_position_text"] = first_pos
            row["first_match_position_ratio"] = first_pos / max(len(text), 1) if np.isfinite(first_pos) else np.nan
            row["no_query_match_in_text"] = int(len(q_set & set(text_tokens)) == 0)

            for limit in [250, 500]:
                prefix_tokens = set(tokenize(text[:limit]))
                row[f"query_text_coverage_first_{limit}"] = len(q_set & prefix_tokens) / max(len(q_set), 1)

            q_yo = set(tokenize(q.replace("ё", "е")))
            t_yo = set(tokenize(text.replace("ё", "е")))
            title_yo = set(tokenize(title.replace("ё", "е")))
            row["query_text_coverage_yo_fold"] = len(q_yo & t_yo) / max(len(q_yo), 1)
            row["query_title_coverage_yo_fold"] = len(q_yo & title_yo) / max(len(q_yo), 1)
            rows.append(row)
        return pd.DataFrame(rows, index=frame.index)


    train_lex = build_lexical_features(train_fe)
    test_lex = build_lexical_features(test_fe)
    train_fe = pd.concat([train_fe, train_lex], axis=1)
    test_fe = pd.concat([test_fe, test_lex], axis=1)

    print("Lexical features added:", len(train_lex.columns))
    display(train_fe[["query", "title", "query_text_coverage", "query_text_jaccard", "query_question_word"]].head(8))

#### Считаем TF-IDF и BM25

In [ ]:
if not CACHE_READY:
    TFIDF_FEATURES = []


    def rowwise_sparse_dot(a, b):
        return np.asarray(a.multiply(b).sum(axis=1)).ravel().astype(np.float32)


    if RUN_TFIDF_BM25:
        from scipy.sparse import csr_matrix
        from sklearn.feature_extraction.text import HashingVectorizer

        all_fe = pd.concat([train_fe, test_fe], ignore_index=True)
        n_train = len(train_fe)
        train_docs = (train_fe["title_norm"].fillna("") + " " + train_fe["text_norm"].fillna("")).tolist()
        all_docs = (all_fe["title_norm"].fillna("") + " " + all_fe["text_norm"].fillna("")).tolist()
        train_queries = train_fe["query_norm"].fillna("").tolist()
        all_queries = all_fe["query_norm"].fillna("").tolist()

        if RUN_WORD_TFIDF:
            try:
                print("Fitting word TF-IDF...")
                word_vectorizer = TfidfVectorizer(
                    token_pattern=r"(?u)\b[0-9a-zа-яё]+\b", lowercase=False,
                    ngram_range=(1, 2), min_df=2, max_features=60000,
                    sublinear_tf=True, dtype=np.float32
                )
                word_vectorizer.fit(pd.Series(train_docs))
                q_word = word_vectorizer.transform(all_queries)
                d_word = word_vectorizer.transform(all_docs)
                word_score = rowwise_sparse_dot(q_word, d_word)
                train_fe["tfidf_word_query_document"] = word_score[:n_train]
                test_fe["tfidf_word_query_document"] = word_score[n_train:]
                TFIDF_FEATURES.append("tfidf_word_query_document")
                print("Word TF-IDF vocabulary:", len(word_vectorizer.vocabulary_))
                del q_word, d_word, word_score, word_vectorizer
                gc.collect()
            except Exception as e:
                print("Word TF-IDF skipped:", repr(e))

        if RUN_CHAR_HASH:
            try:
                print("Calculating hashed character n-gram similarity...")
                char_vectorizer = HashingVectorizer(
                    analyzer="char_wb", ngram_range=(3, 5),
                    n_features=2**18, alternate_sign=False,
                    norm="l2", dtype=np.float32
                )
                q_char = char_vectorizer.transform(all_queries)
                d_char = char_vectorizer.transform(all_docs)
                char_score = rowwise_sparse_dot(q_char, d_char)
                train_fe["char_hash_query_document"] = char_score[:n_train]
                test_fe["char_hash_query_document"] = char_score[n_train:]
                TFIDF_FEATURES.append("char_hash_query_document")
                del q_char, d_char, char_score, char_vectorizer
                gc.collect()
                print("Character hash features created.")
            except Exception as e:
                print("Character hash skipped:", repr(e))

        if RUN_BM25:
            try:
                print("Fitting BM25 vocabulary and calculating sparse scores...")
                count_vectorizer = CountVectorizer(
                    token_pattern=r"(?u)\b[0-9a-zа-яё]+\b", lowercase=False,
                    min_df=2, max_features=60000, dtype=np.float32
                )
                X_doc_train = count_vectorizer.fit_transform(train_docs)
                X_doc_all = count_vectorizer.transform(all_docs).tocsr()
                X_query_all = count_vectorizer.transform(all_queries).tocsr()

                document_frequency = np.asarray((X_doc_train > 0).sum(axis=0)).ravel()
                n_documents = X_doc_train.shape[0]
                idf = np.log(1.0 + (n_documents - document_frequency + 0.5) / (document_frequency + 0.5)).astype(np.float32)

                doc_lengths = np.asarray(X_doc_all.sum(axis=1)).ravel().astype(np.float32)
                avgdl = float(np.mean(np.asarray(X_doc_train.sum(axis=1)).ravel()))
                k1, b = 1.5, 0.75
                row_ids = np.repeat(np.arange(X_doc_all.shape[0]), np.diff(X_doc_all.indptr))
                term_ids = X_doc_all.indices
                term_freq = X_doc_all.data.astype(np.float32)
                norm = k1 * (1.0 - b + b * doc_lengths[row_ids] / max(avgdl, 1.0))
                bm25_data = idf[term_ids] * (term_freq * (k1 + 1.0) / (term_freq + norm))
                X_bm25 = csr_matrix((bm25_data, X_doc_all.indices, X_doc_all.indptr), shape=X_doc_all.shape)
                bm25_score = rowwise_sparse_dot(X_query_all, X_bm25)

                train_fe["bm25_query_document"] = bm25_score[:n_train]
                test_fe["bm25_query_document"] = bm25_score[n_train:]
                TFIDF_FEATURES.append("bm25_query_document")
                print("BM25 vocabulary:", len(count_vectorizer.vocabulary_))
                del X_doc_train, X_doc_all, X_query_all, X_bm25, bm25_score
                del count_vectorizer, row_ids, term_ids, term_freq, bm25_data, norm
                gc.collect()
            except Exception as e:
                print("BM25 skipped:", repr(e))

        del all_fe, train_docs, all_docs, train_queries, all_queries
        gc.collect()

    print("Lexical retrieval features:", TFIDF_FEATURES)
    required_retrieval = {"tfidf_word_query_document", "bm25_query_document"}
    missing_retrieval = required_retrieval - set(TFIDF_FEATURES)
    if missing_retrieval:
        raise RuntimeError(f"Не рассчитаны обязательные TF-IDF/BM25 признаки: {sorted(missing_retrieval)}")

#### Добавляем морфологические признаки Natasha

In [ ]:
if not CACHE_READY:
    NATASHA_FEATURES = []
    HAS_NATASHA = False

    if RUN_NATASHA:
        try:
            from natasha import Segmenter, MorphVocab, NewsEmbedding, NewsMorphTagger, Doc
        except Exception as import_error:
            print("Natasha not installed:", repr(import_error))
            if AUTO_INSTALL_OPTIONAL_PACKAGES:
                try:
                    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "natasha"])
                    from natasha import Segmenter, MorphVocab, NewsEmbedding, NewsMorphTagger, Doc
                except Exception as install_error:
                    print("Natasha installation failed:", repr(install_error))

        try:
            segmenter = Segmenter()
            morph_vocab = MorphVocab()
            morph_embedding = NewsEmbedding()
            morph_tagger = NewsMorphTagger(morph_embedding)

            def lemmatize_text(text):
                doc = Doc(str(text) if text is not None else "")
                doc.segment(segmenter)
                doc.tag_morph(morph_tagger)
                lemmas = []
                for token in doc.tokens:
                    token.lemmatize(morph_vocab)
                    lemma = (token.lemma or token.text or "").lower()
                    if TOKEN_RE.fullmatch(lemma):
                        lemmas.append(lemma)
                return lemmas

            all_text_series = pd.concat([
                train_fe["query_norm"], train_fe["title_norm"], train_fe["text_norm"],
                test_fe["query_norm"], test_fe["title_norm"], test_fe["text_norm"]
            ], ignore_index=True).fillna("").astype(str)
            unique_texts = pd.unique(all_text_series).tolist()
            lemma_cache = {}
            for i, value in enumerate(unique_texts, 1):
                lemma_cache[value] = lemmatize_text(value)
                if i % 5000 == 0:
                    print(f"Lemmatized {i:,}/{len(unique_texts):,} unique texts")

            def lemma_sets(frame):
                q_lemmas = frame["query_norm"].map(lambda x: set(lemma_cache.get(x, [])))
                t_lemmas = frame["title_norm"].map(lambda x: set(lemma_cache.get(x, [])))
                x_lemmas = frame["text_norm"].map(lambda x: set(lemma_cache.get(x, [])))
                return q_lemmas, t_lemmas, x_lemmas

            def add_lemma_features(frame):
                q_lemmas, title_lemmas, text_lemmas = lemma_sets(frame)
                out = frame.copy()
                out["lemma_query_title_coverage"] = [len(q & t) / max(len(q), 1) for q, t in zip(q_lemmas, title_lemmas)]
                out["lemma_query_text_coverage"] = [len(q & t) / max(len(q), 1) for q, t in zip(q_lemmas, text_lemmas)]
                out["lemma_query_title_jaccard"] = [len(q & t) / max(len(q | t), 1) for q, t in zip(q_lemmas, title_lemmas)]
                out["lemma_query_text_jaccard"] = [len(q & t) / max(len(q | t), 1) for q, t in zip(q_lemmas, text_lemmas)]
                out["lemma_query_text_overlap"] = [len(q & t) for q, t in zip(q_lemmas, text_lemmas)]
                return out

            train_fe = add_lemma_features(train_fe)
            test_fe = add_lemma_features(test_fe)
            NATASHA_FEATURES = [
                "lemma_query_title_coverage", "lemma_query_text_coverage",
                "lemma_query_title_jaccard", "lemma_query_text_jaccard",
                "lemma_query_text_overlap"
            ]
            HAS_NATASHA = True
            del lemma_cache, all_text_series, unique_texts
            gc.collect()
            print("Natasha features created:", NATASHA_FEATURES)
        except Exception as e:
            print("Natasha block skipped safely:", repr(e))

    print("HAS_NATASHA:", HAS_NATASHA)

#### Подготавливаем нейросетевые модели

In [ ]:
if not CACHE_READY:
    HAS_TORCH = False
    HAS_TRANSFORMERS = False
    NEURAL_DEVICE = "cpu"

    try:
        import torch
        HAS_TORCH = True
        NEURAL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception as e:
        print("Torch unavailable:", repr(e))

    if HAS_TORCH:
        try:
            from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
            HAS_TRANSFORMERS = True
        except Exception as import_error:
            print("Transformers unavailable:", repr(import_error))
            if AUTO_INSTALL_OPTIONAL_PACKAGES:
                try:
                    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "sentencepiece"])
                    from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
                    HAS_TRANSFORMERS = True
                except Exception as install_error:
                    print("Transformers installation failed:", repr(install_error))

    print("Neural device:", NEURAL_DEVICE)
    if HAS_TORCH and NEURAL_DEVICE == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

    NEURAL_ALLOWED = HAS_TORCH and HAS_TRANSFORMERS and (NEURAL_DEVICE == "cuda" or ALLOW_CPU_NEURAL)

    def autocast_context():
        if HAS_TORCH and NEURAL_DEVICE == "cuda":
            return torch.autocast(device_type="cuda", dtype=torch.float16)
        return torch.autocast(device_type="cpu", enabled=False) if HAS_TORCH else None


    def factorize_texts(series):
        values = series.fillna("").astype(str).tolist()
        codes, uniques = pd.factorize(values, sort=False)
        return values, codes.astype(np.int32), uniques.tolist()


    def encode_unique_transformer(series, prefix, tokenizer, model, max_length=384, batch_size=24):
        values, codes, uniques = factorize_texts(series)
        vectors = []
        model.eval()
        with torch.inference_mode():
            for start in range(0, len(uniques), batch_size):
                batch = [prefix + x for x in uniques[start:start + batch_size]]
                tokens = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
                tokens = {k: v.to(NEURAL_DEVICE) for k, v in tokens.items()}
                with autocast_context():
                    output = model(**tokens)
                hidden = output.last_hidden_state.float()
                mask = tokens["attention_mask"].unsqueeze(-1).float()
                pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
                pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
                vectors.append(pooled.cpu().numpy().astype(np.float32))
                del tokens, output, hidden, mask, pooled
        return np.vstack(vectors), codes


    def rowwise_dense_cosine(q_vecs, q_codes, d_vecs, d_codes, chunk_size=4096):
        result = np.empty(len(q_codes), dtype=np.float32)
        for start in range(0, len(q_codes), chunk_size):
            end = min(start + chunk_size, len(q_codes))
            q = q_vecs[q_codes[start:end]]
            d = d_vecs[d_codes[start:end]]
            result[start:end] = np.sum(q * d, axis=1)
        return result


    NEURAL_FEATURES = []

#### Получаем BGE-M3 признаки

In [ ]:
if not CACHE_READY:
    HAS_BGE_M3 = False

    if RUN_BGE_M3 and NEURAL_ALLOWED:
        try:
            from FlagEmbedding import BGEM3FlagModel
        except Exception as import_error:
            print("FlagEmbedding unavailable:", repr(import_error))
            if AUTO_INSTALL_OPTIONAL_PACKAGES:
                try:
                    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "FlagEmbedding"])
                    from FlagEmbedding import BGEM3FlagModel
                except Exception as install_error:
                    print("FlagEmbedding install failed:", repr(install_error))

        try:
            if "BGEM3FlagModel" in globals():
                all_neural = pd.concat([train_fe, test_fe], ignore_index=True)
                n_train_rows = len(train_fe)
                q_series = all_neural["query_norm"]
                p_series = (all_neural["title_norm"].fillna("") + " " + all_neural["text_norm"].fillna("")).str.strip()

                bge_model = BGEM3FlagModel(BGE_M3_MODEL_ID, use_fp16=(NEURAL_DEVICE == "cuda"))

                def bge_encode_unique(series):
                    _, codes, uniques = factorize_texts(series)
                    dense_parts, sparse_parts = [], []
                    for start in range(0, len(uniques), 512):
                        batch = uniques[start:start + 512]
                        encoded = bge_model.encode(
                            batch,
                            batch_size=BGE_BATCH_SIZE,
                            max_length=BGE_MAX_LENGTH,
                            return_dense=True,
                            return_sparse=True,
                            return_colbert_vecs=False,
                        )
                        dense_parts.append(np.asarray(encoded["dense_vecs"], dtype=np.float32))
                        sparse_parts.extend(encoded["lexical_weights"])
                    return np.vstack(dense_parts), sparse_parts, codes

                q_dense, q_sparse, q_codes = bge_encode_unique(q_series)
                p_dense, p_sparse, p_codes = bge_encode_unique(p_series)
                bge_dense_scores = rowwise_dense_cosine(q_dense, q_codes, p_dense, p_codes)

                def sparse_weight_dot(a, b):
                    if len(a) > len(b):
                        a, b = b, a
                    return sum(float(weight) * float(b.get(token_id, 0.0)) for token_id, weight in a.items())

                bge_sparse_scores = np.empty(len(q_codes), dtype=np.float32)
                for i, (qi, pi) in enumerate(zip(q_codes, p_codes)):
                    bge_sparse_scores[i] = sparse_weight_dot(q_sparse[qi], p_sparse[pi])
                    if (i + 1) % 10000 == 0:
                        print(f"BGE sparse pairs: {i + 1:,}/{len(q_codes):,}")

                train_fe["bge_m3_dense"] = bge_dense_scores[:n_train_rows]
                test_fe["bge_m3_dense"] = bge_dense_scores[n_train_rows:]
                train_fe["bge_m3_sparse"] = bge_sparse_scores[:n_train_rows]
                test_fe["bge_m3_sparse"] = bge_sparse_scores[n_train_rows:]
                NEURAL_FEATURES.extend(["bge_m3_dense", "bge_m3_sparse"])
                HAS_BGE_M3 = True

                del bge_model, q_dense, p_dense, q_sparse, p_sparse, q_codes, p_codes
                del bge_dense_scores, bge_sparse_scores, all_neural
                gc.collect()
                if NEURAL_DEVICE == "cuda":
                    torch.cuda.empty_cache()
                print("BGE-M3 features created.")
        except Exception as e:
            print("BGE-M3 skipped safely:", repr(e))
            gc.collect()
            if HAS_TORCH and NEURAL_DEVICE == "cuda":
                torch.cuda.empty_cache()

    print("HAS_BGE_M3:", HAS_BGE_M3)

#### Получаем BGE Reranker признаки

In [ ]:
if not CACHE_READY:
    HAS_BGE_RERANKER = False

    if RUN_BGE_RERANKER and NEURAL_ALLOWED:
        try:
            all_neural = pd.concat([train_fe, test_fe], ignore_index=True)
            n_train_rows = len(train_fe)
            queries = all_neural["query_norm"].fillna("").astype(str).tolist()
            passages = (
                "Заголовок: " + all_neural["title_norm"].fillna("").astype(str)
                + " Текст фрагмента: " + all_neural["text_norm"].fillna("").astype(str)
            ).tolist()

            rerank_tokenizer = AutoTokenizer.from_pretrained(BGE_RERANKER_MODEL_ID)
            rerank_model = AutoModelForSequenceClassification.from_pretrained(BGE_RERANKER_MODEL_ID)
            rerank_model.to(NEURAL_DEVICE)
            rerank_model.eval()

            def predict_reranker(q_values, p_values, batch_size=RERANKER_BATCH_SIZE):
                scores = np.empty(len(q_values), dtype=np.float32)
                i = 0
                current_bs = batch_size
                while i < len(q_values):
                    end = min(i + current_bs, len(q_values))
                    try:
                        batch_q = q_values[i:end]
                        batch_p = p_values[i:end]
                        tokens = rerank_tokenizer(
                            batch_q, batch_p,
                            padding=True, truncation=True,
                            max_length=RERANKER_MAX_LENGTH,
                            return_tensors="pt"
                        )
                        tokens = {k: v.to(NEURAL_DEVICE) for k, v in tokens.items()}
                        with torch.inference_mode():
                            with autocast_context():
                                logits = rerank_model(**tokens).logits.float()
                        if logits.ndim == 2 and logits.shape[1] > 1:
                            batch_scores = logits[:, -1] - logits[:, 0]
                        else:
                            batch_scores = logits.reshape(-1)
                        scores[i:end] = batch_scores.detach().cpu().numpy()
                        i = end
                        del tokens, logits, batch_scores
                        if i % 2000 < current_bs:
                            print(f"Reranker pairs: {i:,}/{len(q_values):,}")
                    except RuntimeError as e:
                        if "out of memory" in str(e).lower() and current_bs > 2:
                            current_bs = max(2, current_bs // 2)
                            print("CUDA OOM: reduce reranker batch size to", current_bs)
                            if NEURAL_DEVICE == "cuda":
                                torch.cuda.empty_cache()
                            continue
                        raise
                return scores

            reranker_scores = predict_reranker(queries, passages)
            train_fe["bge_reranker_score"] = reranker_scores[:n_train_rows]
            test_fe["bge_reranker_score"] = reranker_scores[n_train_rows:]
            NEURAL_FEATURES.append("bge_reranker_score")
            HAS_BGE_RERANKER = True

            del rerank_model, rerank_tokenizer, reranker_scores, all_neural
            gc.collect()
            if NEURAL_DEVICE == "cuda":
                torch.cuda.empty_cache()
            print("BGE Reranker features created.")
        except Exception as e:
            print("BGE Reranker skipped safely:", repr(e))
            gc.collect()
            if NEURAL_DEVICE == "cuda":
                torch.cuda.empty_cache()

    print("HAS_BGE_RERANKER:", HAS_BGE_RERANKER)

#### Получаем Qwen3-Reranker признаки

In [ ]:
if not CACHE_READY:

    HAS_QWEN_RERANKER = False
    QWEN_FEATURES = []

    if RUN_QWEN_RERANKER:

        import transformers
        from packaging.version import Version
        from transformers import AutoTokenizer, AutoModelForCausalLM

        print("Loading Qwen:", QWEN_MODEL_ID)
        print("Transformers:", transformers.__version__)

        qwen_tokenizer = AutoTokenizer.from_pretrained(
            QWEN_MODEL_ID,
            padding_side="left"
        )
        qwen_tokenizer.padding_side = "left"
        if qwen_tokenizer.pad_token is None:
            qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

        qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_ID,
            torch_dtype=torch.float16
        ).to(NEURAL_DEVICE).eval()
        qwen_model.config.use_cache = False

        QWEN_TASK = "Given a web search query, retrieve relevant passages that answer the query"
        QWEN_PREFIX = (
            "<|im_start|>system\n"
            "Judge whether the Document meets the requirements based on the Query and the Instruct provided. "
            "Note that the answer can only be \"yes\" or \"no\"."
            "<|im_end|>\n<|im_start|>user\n"
        )
        QWEN_SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
        qwen_prefix_tokens = qwen_tokenizer.encode(QWEN_PREFIX, add_special_tokens=False)
        qwen_suffix_tokens = qwen_tokenizer.encode(QWEN_SUFFIX, add_special_tokens=False)
        qwen_yes_ids = qwen_tokenizer("yes", add_special_tokens=False).input_ids
        qwen_no_ids = qwen_tokenizer("no", add_special_tokens=False).input_ids
        assert len(qwen_yes_ids) == 1 and len(qwen_no_ids) == 1, "Qwen yes/no должны быть одиночными токенами."
        qwen_yes_id = qwen_yes_ids[0]
        qwen_no_id = qwen_no_ids[0]

        all_qwen = pd.concat([train_fe, test_fe], ignore_index=True)
        n_train_qwen = len(train_fe)
        qwen_queries_all = all_qwen["query_norm"].fillna("").astype(str).tolist()
        qwen_passages_all = (
            "Заголовок: " + all_qwen["title_norm"].fillna("").astype(str)
            + "\nТекст фрагмента: " + all_qwen["text_norm"].fillna("").astype(str)
        ).tolist()

        # Убираем повторяющиеся пары до inference, затем восстанавливаем исходный порядок.
        qwen_pair_index = pd.MultiIndex.from_arrays(
            [qwen_queries_all, qwen_passages_all], names=["query", "passage"]
        )
        qwen_pair_codes, qwen_unique_pair_index = pd.factorize(qwen_pair_index, sort=False)
        qwen_unique_queries = qwen_unique_pair_index.get_level_values(0).tolist()
        qwen_unique_passages = qwen_unique_pair_index.get_level_values(1).tolist()
        n_qwen_unique = len(qwen_unique_queries)

        print("All train+test pairs:", len(qwen_queries_all))
        print("Unique Qwen pairs:", n_qwen_unique)
        print("Max input tokens:", QWEN_MAX_LENGTH)

        qwen_content_max_length = QWEN_MAX_LENGTH - len(qwen_prefix_tokens) - len(qwen_suffix_tokens)
        if qwen_content_max_length <= 0:
            raise RuntimeError("Qwen prefix/suffix exceed QWEN_MAX_LENGTH.")

        def qwen_predict_batch(batch_queries, batch_passages):
            formatted = [
                f"<Instruct>: {QWEN_TASK}\n<Query>: {query}\n<Document>: {passage}"
                for query, passage in zip(batch_queries, batch_passages)
            ]
            tokenized = qwen_tokenizer(
                formatted,
                padding=False,
                truncation="longest_first",
                return_attention_mask=False,
                max_length=qwen_content_max_length,
            )
            for j, ids in enumerate(tokenized["input_ids"]):
                tokenized["input_ids"][j] = qwen_prefix_tokens + ids + qwen_suffix_tokens

            inputs = qwen_tokenizer.pad(
                tokenized,
                padding=True,
                return_tensors="pt",
                max_length=QWEN_MAX_LENGTH,
            )
            inputs = {key: value.to(NEURAL_DEVICE) for key, value in inputs.items()}

            with torch.inference_mode():
                logits = qwen_model(**inputs, use_cache=False).logits[:, -1, :].float()
                scores = logits[:, qwen_yes_id] - logits[:, qwen_no_id]

            result = scores.detach().cpu().numpy().astype(np.float32)
            del formatted, tokenized, inputs, logits, scores
            return result

        qwen_smoke = qwen_predict_batch(qwen_queries_all[:2], qwen_passages_all[:2])
        assert len(qwen_smoke) == min(2, len(qwen_queries_all))
        assert np.isfinite(qwen_smoke).all()
        print("Qwen smoke test passed. Example scores:", qwen_smoke.tolist())
        del qwen_smoke

        qwen_unique_scores = np.empty(n_qwen_unique, dtype=np.float32)
        qwen_cursor = 0
        qwen_current_batch = QWEN_BATCH_SIZE
        qwen_last_report = 0

        while qwen_cursor < n_qwen_unique:
            qwen_end = min(qwen_cursor + qwen_current_batch, n_qwen_unique)
            try:
                batch_scores = qwen_predict_batch(
                    qwen_unique_queries[qwen_cursor:qwen_end],
                    qwen_unique_passages[qwen_cursor:qwen_end],
                )
                qwen_unique_scores[qwen_cursor:qwen_end] = batch_scores
                qwen_cursor = qwen_end
                del batch_scores

                if qwen_cursor - qwen_last_report >= 1000 or qwen_cursor == n_qwen_unique:
                    print(f"Qwen scored: {qwen_cursor:,}/{n_qwen_unique:,} | batch_size={qwen_current_batch}")
                    qwen_last_report = qwen_cursor

            except RuntimeError as qwen_error:
                is_oom = "out of memory" in str(qwen_error).lower()
                if is_oom and qwen_current_batch > 1:
                    qwen_current_batch = max(1, qwen_current_batch // 2)
                    print("Qwen CUDA OOM — reducing batch size to", qwen_current_batch)
                    torch.cuda.empty_cache()
                    gc.collect()
                    continue
                raise

        qwen_all_scores = qwen_unique_scores[qwen_pair_codes]
        assert len(qwen_all_scores) == len(all_qwen)
        assert np.isfinite(qwen_all_scores).all(), "Qwen produced non-finite scores."

        train_fe["qwen_reranker_score"] = qwen_all_scores[:n_train_qwen]
        test_fe["qwen_reranker_score"] = qwen_all_scores[n_train_qwen:]
        QWEN_FEATURES = ["qwen_reranker_score"]
        NEURAL_FEATURES.extend(QWEN_FEATURES)
        HAS_QWEN_RERANKER = True

        print("Qwen feature created:", QWEN_FEATURES)
        print("Qwen score range:", float(np.min(qwen_all_scores)), "—", float(np.max(qwen_all_scores)))
        print("HAS_QWEN_RERANKER:", HAS_QWEN_RERANKER)

        # Сохраняем полный score-вектор: пригодится, если потребуется повторить только CV.
        np.save(OUTPUT_DIR / "qwen3_reranker_scores_train_test.npy", qwen_all_scores)
        del qwen_model, qwen_tokenizer, qwen_unique_scores, qwen_all_scores
        del qwen_pair_index, qwen_pair_codes, qwen_unique_pair_index
        del qwen_unique_queries, qwen_unique_passages, qwen_queries_all, qwen_passages_all, all_qwen
        gc.collect()
        torch.cuda.empty_cache()

#### Собираем базовые 63 признака

In [ ]:
if not CACHE_READY:
    RANK_BASE_FEATURES = [
        "query_text_coverage", "query_text_jaccard", "query_title_coverage",
        "query_title_jaccard", "passage_position_ratio_zero_to_last",
        "text_to_article_char_ratio", "tfidf_word_query_document",
        "char_hash_query_document", "bm25_query_document",
        "lemma_query_text_coverage", "lemma_query_text_jaccard",
        "bge_m3_dense", "bge_m3_sparse", "bge_reranker_score", "qwen_reranker_score", "e5_query_document", "sbert_ru_query_document"
    ]

    for frame in [train_fe, test_fe]:
        for col in RANK_BASE_FEATURES:
            if col in frame.columns:
                frame[f"{col}_query_rank"] = frame.groupby("query_id")[col].rank(pct=True, method="average")

    CATEGORICAL_FEATURES = [c for c in ["ingest_source", "query_question_word"] if c in train_fe.columns]
    EXCLUDE_COLUMNS = {
        "id", "query_id", "document_id", "article_id", "relevance",
        "query", "title", "text",
        "query_norm", "title_norm", "text_norm",
        "query_yo_fold", "title_yo_fold", "text_yo_fold",
        "passage_index", "article_passage_count"
    }

    feature_cols = [
        c for c in train_fe.columns
        if c not in EXCLUDE_COLUMNS and c in test_fe.columns
        and (pd.api.types.is_numeric_dtype(train_fe[c]) or c in CATEGORICAL_FEATURES)
    ]

    constant_features = [
        c for c in feature_cols
        if c not in CATEGORICAL_FEATURES and train_fe[c].nunique(dropna=True) <= 1
    ]
    feature_cols = [c for c in feature_cols if c not in constant_features]

    print("Features:", len(feature_cols))
    print("Categorical:", [c for c in CATEGORICAL_FEATURES if c in feature_cols])
    print("Removed constants:", constant_features)
    print("Neural features present:", [c for c in NEURAL_FEATURES if c in feature_cols])

    REQUIRED_FEATURES = {
        "tfidf_word_query_document", "bm25_query_document",
        "bge_m3_dense", "bge_m3_sparse", "bge_reranker_score",
        "qwen_reranker_score", "qwen_reranker_score_query_rank",
        "lemma_query_title_coverage", "lemma_query_text_coverage",
        "lemma_query_title_jaccard", "lemma_query_text_jaccard",
        "lemma_query_text_overlap",
    }
    missing_required = REQUIRED_FEATURES - set(feature_cols)
    if missing_required:
        raise RuntimeError(f"Не восстановлены обязательные признаки: {sorted(missing_required)}")
    if len(feature_cols) != 63:
        raise RuntimeError(
            f"Ожидалось 63 признака (61 базовый + 2 Qwen), получено {len(feature_cols)}. "
            "Не продолжай с неполной матрицей: проверь Natasha, TF-IDF/BM25, BGE-M3, BGE Reranker и Qwen."
        )
    assert HAS_QWEN_RERANKER and set(["qwen_reranker_score", "qwen_reranker_score_query_rank"]).issubset(feature_cols)
    print("\nQWEN EXPERIMENT FEATURE CHECK PASSED: ровно 63 признака (61 + 2 Qwen).")

#### Формируем 35 контекстных признаков

In [ ]:
if not CACHE_READY:
    import re
    import unicodedata

    ENSEMBLE_MODEL_KEYS = ['bge_reranker_score','bge_m3_dense','bge_m3_sparse','qwen_reranker_score']
    ANSWER_YEAR_RE = re.compile(r'(?<!\d)(?:1[0-9]{3}|20[0-9]{2})(?!\d)')
    WORD_RE = re.compile(r'[а-яёa-z0-9]+',flags=re.IGNORECASE)


    def norm_light(s):
        if pd.isna(s): return ''
        s=unicodedata.normalize('NFKC',str(s)).lower().replace('ё','е')
        s=''.join(c for c in s if unicodedata.category(c)!='Mn')
        return ' '.join(s.split())


    def add_ranking_context_features(frame):
        frame=frame.copy()
        group=frame.groupby('query_id',sort=False)
        result=pd.DataFrame(index=frame.index)
        base_names=[]

        for name in ENSEMBLE_MODEL_KEYS:
            val=pd.to_numeric(frame[name],errors='coerce').replace([np.inf,-np.inf],np.nan)
            means=val.groupby(frame['query_id']).transform('mean')
            med=val.groupby(frame['query_id']).transform('median')
            maxi=val.groupby(frame['query_id']).transform('max')
            std=val.groupby(frame['query_id']).transform('std').replace(0,np.nan)
            pref='ctx_'+name
            result[pref+'_minus_mean']=(val-means).astype('float32')
            result[pref+'_minus_median']=(val-med).astype('float32')
            result[pref+'_to_best']=(maxi-val).astype('float32')
            result[pref+'_z']=((val-means)/std).fillna(0).clip(-10,10).astype('float32')
            base_names.extend([pref+'_minus_mean',pref+'_minus_median',pref+'_to_best',pref+'_z'])

        ranks={k:pd.to_numeric(frame[k],errors='coerce').groupby(frame['query_id']).rank(pct=True)
               for k in ENSEMBLE_MODEL_KEYS}
        result['ctx_qwen_minus_bge_rank']=(ranks['qwen_reranker_score']-ranks['bge_reranker_score']).astype('float32')
        result['ctx_dense_minus_reranker_rank']=(ranks['bge_m3_dense']-ranks['bge_reranker_score']).astype('float32')
        result['ctx_neural_rank_std']=pd.DataFrame(ranks).std(axis=1).astype('float32')
        result['ctx_neural_rank_mean']=pd.DataFrame(ranks).mean(axis=1).astype('float32')

        article_group = frame.groupby(['query_id','article_id'],sort=False,dropna=False)
        result['ctx_same_article_candidates']=article_group['id'].transform('size').astype('float32')
        for score in ['bge_reranker_score','qwen_reranker_score']:
            result['ctx_article_best_'+score]=(article_group[score].transform('max')-frame[score]).astype('float32')
            result['ctx_article_mean_'+score]=(article_group[score].transform('mean')).astype('float32')

        if 'ingest_source' in frame.columns:
            result['ctx_same_source_candidates']=frame.groupby(['query_id','ingest_source'],dropna=False)['id'].transform('size').astype('float32')
            result['ctx_source_fraction']=(result['ctx_same_source_candidates'] / group['id'].transform('size')).astype('float32')

        if 'query' in frame and 'text' in frame:
            q=frame['query'].fillna('').map(norm_light)
            t=frame['text'].fillna('').map(norm_light)
            title=frame['title'].fillna('').map(norm_light) if 'title' in frame else pd.Series('',index=frame.index)
            query_year=q.map(lambda x:set(ANSWER_YEAR_RE.findall(x)))
            doc_year=(title+' '+t).map(lambda x:set(ANSWER_YEAR_RE.findall(x)))
            result['ctx_query_has_year']=q.map(lambda x:int(bool(ANSWER_YEAR_RE.search(x)))).astype('float32')
            result['ctx_doc_has_year']=(title+' '+t).map(lambda x:int(bool(ANSWER_YEAR_RE.search(x)))).astype('float32')
            result['ctx_year_overlap']=[float(len(a&b)) for a,b in zip(query_year,doc_year)]
            result['ctx_query_asks_when']=q.str.contains(r'\b(когда|год|году|лет|веке|дата|дату)\b',regex=True).astype('float32')
            result['ctx_query_asks_where']=q.str.contains(r'\b(где|куда|откуда|место|находится)\b',regex=True).astype('float32')
            result['ctx_query_asks_who']=q.str.contains(r'\b(кто|кому|кем|чей|чья|кого)\b',regex=True).astype('float32')
            result['ctx_query_asks_how_many']=q.str.contains(r'\b(сколько|число|количество)\b',regex=True).astype('float32')

            def first_window_coverage(a,b):
                toks=set(WORD_RE.findall(a))
                return len(toks&set(WORD_RE.findall(b[:320])))/max(len(toks),1)
            result['ctx_first_320_coverage']=[first_window_coverage(a,b) for a,b in zip(q,t)]

        assert len(result)==len(frame)
        return result.replace([np.inf,-np.inf],np.nan),result.columns.tolist()

    ctx_train, RICH_FEATURE_NAMES=add_ranking_context_features(train_fe)
    ctx_test, rich_cols_test=add_ranking_context_features(test_fe)
    assert RICH_FEATURE_NAMES==rich_cols_test
    for c in RICH_FEATURE_NAMES:
        train_fe[c]=ctx_train[c].to_numpy()
        test_fe[c]=ctx_test[c].to_numpy()

    RICH_FEATURES=MODEL_FEATURES+RICH_FEATURE_NAMES
    print('Champion63 features:',len(MODEL_FEATURES))
    print('New context features:',len(RICH_FEATURE_NAMES))
    print('Rich features:',len(RICH_FEATURES))
    print('New feature names:',RICH_FEATURE_NAMES)
    assert not set(RICH_FEATURE_NAMES)&set(MODEL_FEATURES)
    assert train_fe['id'].is_unique and test_fe['id'].is_unique

In [ ]:
if not CACHE_READY:
    assert len(MODEL_FEATURES) == 63
    assert len(RICH_FEATURE_NAMES) == 35
    RICH_FEATURES = MODEL_FEATURES + RICH_FEATURE_NAMES
    assert len(RICH_FEATURES) == 98
    assert len(set(RICH_FEATURES)) == 98

    train_fe[["id", "query_id", "document_id", "article_id", "relevance"] + RICH_FEATURES].to_csv(
        OUTPUT_DIR / "train_features_exp11_rich98.csv", index=False, float_format="%.9g"
    )
    test_fe[["id", "query_id", "document_id", "article_id"] + RICH_FEATURES].to_csv(
        OUTPUT_DIR / "test_features_exp11_rich98.csv", index=False, float_format="%.9g"
    )
    CACHE_SOURCE = "computed from raw train/test"
    CACHE_READY = True
    print("Rich98 created and saved:", len(RICH_FEATURES))

CAT_COLS = [c for c in ["ingest_source", "query_question_word"] if c in RICH_FEATURES]
assert len(RICH_FEATURES) == 98
assert len(RICH_FEATURE_NAMES) == 35
assert set(CANDIDATE_RANK_SIGNALS).issubset(RICH_FEATURES)
print("Rich98 ready for Jina:", len(RICH_FEATURES))

## 4. Добавляем Jina v3.5

In [ ]:
# проверяем исходные тексты и идентификаторы

In [ ]:
assert len(MODEL_FEATURES) == 63
assert len(RICH_FEATURES) == 98
assert len(RICH_FEATURE_NAMES) == 35
assert all(col in train_fe.columns and col in test_fe.columns for col in RICH_FEATURES)

print("Rich98 сформирован.")
print("Total features:", len(RICH_FEATURES))
print("Original 63-feature component:", len(RICH_FEATURES) - len(RICH_FEATURE_NAMES))
print("Context features:", len(RICH_FEATURE_NAMES))
print("Feature sample:", RICH_FEATURES[:10])

print("Candidate ranking signals present:")
for signal in CANDIDATE_RANK_SIGNALS:
    print(" -", signal, "| train non-null:", int(train_fe[signal].notna().sum()),
          "| test non-null:", int(test_fe[signal].notna().sum()))

# Исходные текстовые поля используются без изменения.
TEXT_COLUMNS = ["query", "title", "text"]
SPECIAL_CHARS = {
    "NBSP": "\u00A0",
    "NARROW_NBSP": "\u202F",
    "THIN_SPACE": "\u2009",
    "ZERO_WIDTH_SPACE": "\u200B",
    "WORD_JOINER": "\u2060",
    "BOM": "\uFEFF",
}
unicode_rows = []
raw_quality_rows = []

for split_name, frame in [("train", train), ("test", test)]:
    for col in TEXT_COLUMNS:
        s = frame[col].fillna("").astype(str)
        rec = {
            "split": split_name,
            "column": col,
            "rows": len(frame),
            "missing": int(frame[col].isna().sum()),
            "empty_after_strip": int((s.str.strip() == "").sum()),
            "rows_with_combining_marks": int(s.str.contains(r"[\u0300-\u036f]", regex=True).sum()),
        }
        for name, char in SPECIAL_CHARS.items():
            rec[f"rows_with_{name.lower()}"] = int(s.str.contains(re.escape(char), regex=True).sum())
        unicode_rows.append(rec)

    q_lengths = frame["query"].fillna("").astype(str).str.len()
    t_lengths = frame["text"].fillna("").astype(str).str.len()
    title_lengths = frame["title"].fillna("").astype(str).str.len()
    raw_quality_rows.append({
        "split": split_name,
        "rows": len(frame),
        "unique_id": int(frame["id"].astype(str).nunique()),
        "duplicate_id": int(frame["id"].astype(str).duplicated().sum()),
        "missing_query": int(frame["query"].isna().sum()),
        "missing_title": int(frame["title"].isna().sum()),
        "missing_text": int(frame["text"].isna().sum()),
        "query_len_median": float(q_lengths.median()),
        "text_len_median": float(t_lengths.median()),
        "text_len_p95": float(t_lengths.quantile(0.95)),
        "text_len_max": int(t_lengths.max()),
        "text_over_4000_chars": int((t_lengths > 4000).sum()),
        "title_len_median": float(title_lengths.median()),
        "query_id_count": int(frame["query_id"].nunique()),
        "max_queries_per_query_id": int(frame.groupby("query_id")["query"].nunique(dropna=False).max()),
    })

unicode_audit = pd.DataFrame(unicode_rows)
raw_quality_audit = pd.DataFrame(raw_quality_rows)

print("Raw text / Unicode audit (no values modified):")
display(unicode_audit)
print("Raw data quality and text-length audit:")
display(raw_quality_audit)

unicode_audit.to_csv(OUTPUT_DIR / "exp13_4_jina_unicode_audit.csv", index=False)
raw_quality_audit.to_csv(OUTPUT_DIR / "exp13_4_jina_raw_quality_audit.csv", index=False)

assert train["id"].astype(str).is_unique and test["id"].astype(str).is_unique
assert set(train_fe["id"].astype(str)) == set(train["id"].astype(str))
assert set(test_fe["id"].astype(str)) == set(test["id"].astype(str))
assert train["relevance"].isin([0, 1]).all()
assert train.groupby("query_id")["query"].nunique(dropna=False).max() == 1
assert test.groupby("query_id")["query"].nunique(dropna=False).max() == 1
print("Raw source audit passed. исходные признаки не изменялись.")

### Отбор кандидатов

In [ ]:
missing_signals = set(CANDIDATE_RANK_SIGNALS) - set(train_fe.columns)
missing_signals |= set(CANDIDATE_RANK_SIGNALS) - set(test_fe.columns)
assert not missing_signals, f"В кэше не хватает сигналов для отбора Top-30: {missing_signals}"

def select_topk_mask(frame, top_k=30):
    q = frame["query_id"].astype(str).reset_index(drop=True)
    rank_frame = pd.DataFrame(index=np.arange(len(frame)))

    for col in CANDIDATE_RANK_SIGNALS:
        values = pd.to_numeric(frame[col], errors="coerce").reset_index(drop=True)
        rank_frame[col] = values.groupby(q).rank(
            method="average", pct=True, na_option="bottom"
        )

    # Средний процентильный ранг: одинаковый вес каждому существующему сигналу.
    consensus = rank_frame.mean(axis=1).fillna(0.0).to_numpy(dtype=np.float64)
    order_df = pd.DataFrame({
        "row": np.arange(len(frame)),
        "query": q.to_numpy(),
        "consensus": consensus,
        "id": frame["id"].astype(str).to_numpy(),
    })
    order_df = order_df.sort_values(
        ["query", "consensus", "id"],
        ascending=[True, False, True],
        kind="mergesort",
    )
    order_df["rank_in_query"] = order_df.groupby("query", sort=False).cumcount()
    selected_rows = order_df.loc[order_df["rank_in_query"] < int(top_k), "row"].to_numpy()

    mask = np.zeros(len(frame), dtype=bool)
    mask[selected_rows] = True
    return mask, consensus

train_top30_mask, train_consensus = select_topk_mask(train_fe, TOP_K)
test_top30_mask, test_consensus = select_topk_mask(test_fe, TOP_K)

train_selected_idx = np.flatnonzero(train_top30_mask)
test_selected_idx = np.flatnonzero(test_top30_mask)

train_positive = train_fe["relevance"].to_numpy() > 0
total_positive = int(train_positive.sum())
selected_positive = int((train_top30_mask & train_positive).sum())
positive_recall_at_topk = selected_positive / max(total_positive, 1)

per_query_coverage = []
for q, g in train_fe.groupby("query_id", sort=False):
    pos = g["relevance"].to_numpy() > 0
    if pos.any():
        selected = train_top30_mask[g.index.to_numpy()]
        per_query_coverage.append(float((selected & pos).sum() / pos.sum()))

candidate_counts_train = train_fe.groupby("query_id", sort=False).size()
candidate_counts_test = test_fe.groupby("query_id", sort=False).size()

selection_report = pd.DataFrame([
    {"split": "train", "rows": len(train_fe), "selected": int(train_top30_mask.sum()),
     "selected_share": float(train_top30_mask.mean()), "queries": train_fe.query_id.nunique()},
    {"split": "test", "rows": len(test_fe), "selected": int(test_top30_mask.sum()),
     "selected_share": float(test_top30_mask.mean()), "queries": test_fe.query_id.nunique()},
])
display(selection_report.round(4))

print(f"Train positive-row recall@{TOP_K}: {positive_recall_at_topk:.4%}")
print(f"Queries with all positive rows inside Top-{TOP_K}: "
      f"{np.mean(np.asarray(per_query_coverage) == 1.0):.2%}")
print("Top-30 coverage by query — mean:", round(float(np.mean(per_query_coverage)), 4))
print("Candidate counts per query — train median/max:",
      int(candidate_counts_train.median()), int(candidate_counts_train.max()))
print("Candidate counts per query — test median/max:",
      int(candidate_counts_test.median()), int(candidate_counts_test.max()))

if positive_recall_at_topk < 0.95:
    print(
        "Top-K отсекает заметную долю положительных строк. "
        "Проверь TOP_K перед запуском Jina."
    )
else:
    print("Top-K coverage looks acceptable for the first Lite run.")

selection_report.to_csv(OUTPUT_DIR / "exp13_4_jina_topk_selection_report.csv", index=False)

### Подготовка модели

In [ ]:
import torch
import transformers
from transformers import AutoModel, AutoTokenizer
from packaging.version import Version

assert Version(transformers.__version__) >= Version("4.57.3"), (
    f"Требуется Transformers >= 4.57.3, "
    f"установлена версия {transformers.__version__}"
)

assert torch.cuda.is_available(), "Для запуска необходимо включить GPU."

device = torch.device("cuda")

free_bytes, total_bytes = torch.cuda.mem_get_info(0)

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Свободная видеопамять: {free_bytes / 2**30:.2f} / {total_bytes / 2**30:.2f} GiB")
print("Среда готова к запуску.")

### Подготовка запросов и документов

In [ ]:
def jina_normalize_exp11(value, fold_yo=False):
    if pd.isna(value):
        return ""
    s = html.unescape(str(value))
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"[\u0300-\u036f]", "", s)
    for char in [
        "\u00A0", "\u202F", "\u2009", "\u200A",
        "\u200B", "\u2060", "\uFEFF"
    ]:
        s = s.replace(char, " ")
    s = re.sub(r"\s+", " ", s).strip().lower()
    if fold_yo:
        s = s.replace("ё", "е")
    return s


def jina_normalize_simple(value):

    if pd.isna(value):
        return ""
    s = html.unescape(str(value))
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"[\u200B\u2060\uFEFF]", "", s)
    return re.sub(r"\s+", " ", s).strip()


def jina_make_document(title, text, normalizer=jina_normalize_exp11):
    title_norm = normalizer(title)
    text_norm = normalizer(text)
    return f"Заголовок: {title_norm}\nТекст фрагмента: {text_norm}"


def build_jina_tasks(frame_features, raw_frame, selected_mask, split_name, consensus):
    tasks = []
    selected_rows = np.flatnonzero(selected_mask)
    if len(selected_rows) == 0:
        return tasks

    raw_ids = raw_frame["id"].astype(str).reset_index(drop=True)
    feature_ids = frame_features["id"].astype(str).reset_index(drop=True)
    assert raw_ids.equals(feature_ids), f"{split_name}: raw/cache row order mismatch"

    selected_meta = pd.DataFrame({
        "row_idx": selected_rows,
        "query_id": frame_features.iloc[selected_rows]["query_id"].astype(str).to_numpy(),
        "consensus": np.asarray(consensus)[selected_rows],
        "id": frame_features.iloc[selected_rows]["id"].astype(str).to_numpy(),
    })

    for query_id, group in selected_meta.groupby("query_id", sort=False):
        group = group.sort_values(
            ["consensus", "id"], ascending=[False, True], kind="mergesort"
        )
        row_indices = group["row_idx"].to_numpy(dtype=np.int64)

        raw_query_values = raw_frame.iloc[row_indices]["query"].map(jina_normalize_exp11).unique()
        assert len(raw_query_values) == 1, (
            f"{split_name}, query_id={query_id}: more than one normalized query string"
        )
        query = raw_query_values[0]

        docs = [
            jina_make_document(title, text, normalizer=jina_normalize_exp11)
            for title, text in zip(
                raw_frame.iloc[row_indices]["title"],
                raw_frame.iloc[row_indices]["text"],
            )
        ]
        assert len(docs) == len(row_indices) and all(isinstance(d, str) for d in docs)

        tasks.append({
            "split": split_name,
            "query_id": str(query_id),
            "query": query,
            "row_indices": row_indices,
            "documents": docs,
        })
    return tasks


jina_tasks = (
    build_jina_tasks(train_fe, train, train_top30_mask, "train", train_consensus)
    + build_jina_tasks(test_fe, test, test_top30_mask, "test", test_consensus)
)

offset = 0
for task in jina_tasks:
    task["start"] = offset
    task["end"] = offset + len(task["row_indices"])
    offset = task["end"]

JINA_TOTAL_SELECTED_ROWS = offset
JINA_NUM_TASKS = len(jina_tasks)
assert JINA_NUM_TASKS > 0 and JINA_TOTAL_SELECTED_ROWS > 0



def jina_fingerprint():
    h = hashlib.sha256()
    meta = (
        f"{JINA_MODEL_ID}|revision={JINA_MODEL_REVISION}|topk={TOP_K}|"
        f"normalization={JINA_INPUT_NORMALIZATION}|"
        f"max_doc_tokens_per_doc={JINA_MAX_DOC_TOKENS_PER_DOC}|"
        f"total_doc_token_budget={JINA_TOTAL_DOC_TOKEN_BUDGET}|"
        f"max_query_tokens={JINA_MAX_QUERY_TOKENS}|"
        f"truncation={JINA_TOKEN_TRUNCATION}|"
        + "|".join(CANDIDATE_RANK_SIGNALS)
    )
    h.update(meta.encode("utf-8"))
    for task in jina_tasks:
        h.update(task["split"].encode("utf-8"))
        h.update(task["query_id"].encode("utf-8"))
        h.update(task["query"].encode("utf-8"))
        for row_idx, doc in zip(task["row_indices"], task["documents"]):
            frame_id = (
                train_fe.iloc[row_idx]["id"]
                if task["split"] == "train"
                else test_fe.iloc[row_idx]["id"]
            )
            h.update(str(frame_id).encode("utf-8"))
            h.update(b"\x00")
            h.update(doc.encode("utf-8"))
            h.update(b"\x01")
    return h.hexdigest()


JINA_FP = jina_fingerprint()
JINA_CHECKPOINT_NAME = "exp13_4_jina_v35_lowvram_checkpoint.npz"
JINA_COMPLETE_NAME = "exp13_4_jina_v35_lowvram_complete.npz"
JINA_CHECKPOINT_FILE = OUTPUT_DIR / JINA_CHECKPOINT_NAME
JINA_COMPLETE_FILE = OUTPUT_DIR / JINA_COMPLETE_NAME


def jina_save_checkpoint(path, scores, done_tasks):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("wb") as handle:
        np.savez_compressed(
            handle,
            scores=np.asarray(scores, dtype=np.float32),
            done_tasks=np.asarray(int(done_tasks), dtype=np.int32),
            fingerprint=np.asarray(JINA_FP),
        )
    os.replace(tmp, path)


def jina_load_checkpoint(path):
    path = Path(path)
    if not path.is_file():
        return None
    try:
        with np.load(path, allow_pickle=False) as data:
            fp = str(data["fingerprint"].item())
            scores = np.asarray(data["scores"], dtype=np.float32)
            done_tasks = int(data["done_tasks"].item())
        if fp != JINA_FP or scores.shape != (JINA_TOTAL_SELECTED_ROWS,):
            return None
        if not (0 <= done_tasks <= JINA_NUM_TASKS):
            return None
        done_rows = jina_tasks[done_tasks - 1]["end"] if done_tasks > 0 else 0
        if not np.isfinite(scores[:done_rows]).all():
            return None
        if not np.isnan(scores[done_rows:]).all():
            return None
        return scores, done_tasks
    except Exception as exc:
        print("Could not read Jina checkpoint:", path, repr(exc))
        return None


checkpoint_paths = [JINA_CHECKPOINT_FILE, JINA_COMPLETE_FILE]
for root in (OUTPUT_DIR,):
    if root.exists():
        for filename in (JINA_CHECKPOINT_NAME, JINA_COMPLETE_NAME):
            try:
                checkpoint_paths.extend(root.rglob(filename))
            except (PermissionError, OSError):
                pass

valid_checkpoints = []
for path in dict.fromkeys(checkpoint_paths):
    loaded = jina_load_checkpoint(path)
    if loaded is not None:
        scores, done = loaded
        valid_checkpoints.append((done, str(path), scores))

if valid_checkpoints:
    valid_checkpoints.sort(key=lambda x: x[0], reverse=True)
    JINA_SCORES_FLAT = valid_checkpoints[0][2]
    JINA_DONE_TASKS = valid_checkpoints[0][0]
    JINA_RESUME_SOURCE = valid_checkpoints[0][1]
    print("Compatible memory-safe Jina checkpoint:", JINA_DONE_TASKS, "/", JINA_NUM_TASKS)
    print("Source:", JINA_RESUME_SOURCE)
else:
    JINA_SCORES_FLAT = np.full(JINA_TOTAL_SELECTED_ROWS, np.nan, dtype=np.float32)
    JINA_DONE_TASKS = 0
    JINA_RESUME_SOURCE = None
    print("No compatible memory-safe checkpoint. Starting from zero.")

print("Train Top-30 queries:", sum(t["split"] == "train" for t in jina_tasks))
print("Test Top-30 queries:", sum(t["split"] == "test" for t in jina_tasks))
print("Selected candidate rows:", JINA_TOTAL_SELECTED_ROWS)
print("Jina listwise calls:", JINA_NUM_TASKS)
print("Main input normalization:", JINA_INPUT_NORMALIZATION)
print("Per-document cap:", JINA_MAX_DOC_TOKENS_PER_DOC)
print("Per-query document token budget:", JINA_TOTAL_DOC_TOKEN_BUDGET)
print("Query cap:", JINA_MAX_QUERY_TOKENS)
print("Fingerprint:", JINA_FP[:24] + "...")

### Оценки Jina

In [ ]:
JINA_COMPLETE = False

if JINA_DONE_TASKS >= JINA_NUM_TASKS:
    JINA_COMPLETE = True
    print("All memory-safe Jina scores already exist. GPU inference is skipped.")
else:
    if not RUN_JINA:
        raise RuntimeError("RUN_JINA=False, но полного Jina-кэша нет.")

    from transformers import AutoModel, AutoTokenizer

    if not torch.cuda.is_available():
        raise RuntimeError("Для Jina включи GPU в настройках среды.")

    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print("Loading", JINA_MODEL_ID, "in", dtype, "...")

    jina_tokenizer = AutoTokenizer.from_pretrained(
        JINA_MODEL_ID,
        revision=JINA_MODEL_REVISION,
        trust_remote_code=True,
    )
    model = AutoModel.from_pretrained(
        JINA_MODEL_ID,
        revision=JINA_MODEL_REVISION,
        trust_remote_code=True,
        torch_dtype=dtype,
    ).eval().to("cuda")

    model._tokenizer = jina_tokenizer
    if model._tokenizer.pad_token is None:
        model._tokenizer.pad_token = model._tokenizer.unk_token
        model._tokenizer.pad_token_id = model._tokenizer.convert_tokens_to_ids(
            model._tokenizer.pad_token
        )
    model._tokenizer.padding_side = "left"

    torch.backends.cuda.matmul.allow_tf32 = True
    print("Model loaded:", next(model.parameters()).device)
    free_gb, total_gb = torch.cuda.mem_get_info(0)
    print(f"Initial VRAM free: {free_gb / 2**30:.2f} / {total_gb / 2**30:.2f} GiB")
    print(f"Model allocated: {torch.cuda.memory_allocated(0) / 2**30:.2f} GiB")

    memory_telemetry_rows = []

    def jina_gpu_memory_snapshot():
        torch.cuda.synchronize(0)
        free_bytes, total_bytes = torch.cuda.mem_get_info(0)
        return {
            "allocated_gb": float(torch.cuda.memory_allocated(0) / 2**30),
            "reserved_gb": float(torch.cuda.memory_reserved(0) / 2**30),
            "free_gb": float(free_bytes / 2**30),
            "total_gb": float(total_bytes / 2**30),
            "peak_allocated_gb": float(torch.cuda.max_memory_allocated(0) / 2**30),
            "peak_reserved_gb": float(torch.cuda.max_memory_reserved(0) / 2**30),
        }

    def jina_cleanup_memory():
        gc.collect()
        torch.cuda.synchronize(0)
        torch.cuda.empty_cache()
        if JINA_CUDA_IPC_COLLECT and hasattr(torch.cuda, "ipc_collect"):
            try:
                torch.cuda.ipc_collect()
            except Exception as cleanup_exc:
                print("CUDA IPC cleanup skipped:", repr(cleanup_exc))
        torch.cuda.synchronize(0)
        return jina_gpu_memory_snapshot()

    def jina_record_memory(event, done_tasks, before=None, after=None,
                           query_peak_allocated_gb=None,
                           query_peak_reserved_gb=None, note=""):
        row = {
            "event": str(event),
            "done_queries": int(done_tasks),
            "elapsed_sec": (
                float(time.monotonic() - session_start)
                if "session_start" in globals() else None
            ),
            "query_peak_allocated_gb": query_peak_allocated_gb,
            "query_peak_reserved_gb": query_peak_reserved_gb,
            "note": str(note),
        }
        if before:
            row.update({"before_" + k: v for k, v in before.items()})
        if after:
            row.update({"after_" + k: v for k, v in after.items()})
        memory_telemetry_rows.append(row)
        pd.DataFrame(memory_telemetry_rows).to_csv(
            JINA_MEMORY_TELEMETRY_FILE, index=False
        )
        return row

    initial_memory = jina_gpu_memory_snapshot()
    jina_record_memory(
        "model_loaded",
        JINA_DONE_TASKS,
        after=initial_memory,
        note="Jina model remains resident on GPU during inference.",
    )
    print(
        "Memory after model load | "
        f"allocated={initial_memory['allocated_gb']:.2f} GiB | "
        f"reserved={initial_memory['reserved_gb']:.2f} GiB | "
        f"free={initial_memory['free_gb']:.2f} GiB"
    )

    def _encode_no_special(text):
        return jina_tokenizer.encode(
            str(text),
            add_special_tokens=False,
        )

    def _decode_tokens(token_ids):
        return jina_tokenizer.decode(
            token_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

    def _waterfill_caps(lengths, budget):
        desired = [
            min(int(length), int(JINA_MAX_DOC_TOKENS_PER_DOC))
            for length in lengths
        ]
        if sum(desired) <= int(budget):
            return desired

        caps = [0] * len(desired)
        active = [i for i, length in enumerate(desired) if length > 0]
        remaining = int(budget)

        while active and remaining > 0:
            fair_share = remaining // len(active)
            if fair_share <= 0:
                for i in active[:remaining]:
                    caps[i] = 1
                break

            short = [i for i in active if desired[i] <= fair_share]
            if short:
                for i in short:
                    caps[i] = desired[i]
                    remaining -= desired[i]
                short_set = set(short)
                active = [i for i in active if i not in short_set]
            else:
                base = fair_share
                for i in active:
                    caps[i] = base
                remainder = remaining - base * len(active)
                for i in active[:remainder]:
                    caps[i] += 1
                break

        return caps

    def jina_prepare_inputs(query, documents):
        query_ids = _encode_no_special(query)
        query_was_truncated = len(query_ids) > JINA_MAX_QUERY_TOKENS
        if query_was_truncated:
            query_ids = query_ids[:JINA_MAX_QUERY_TOKENS]
        prepared_query = _decode_tokens(query_ids)

        doc_token_ids = [_encode_no_special(doc) for doc in documents]
        original_lengths = [len(ids) for ids in doc_token_ids]
        caps = _waterfill_caps(original_lengths, JINA_TOTAL_DOC_TOKEN_BUDGET)

        prepared_documents = [
            _decode_tokens(ids[:cap]) if cap > 0 else ""
            for ids, cap in zip(doc_token_ids, caps)
        ]
        used_lengths = [min(len(ids), cap) for ids, cap in zip(doc_token_ids, caps)]

        stats = {
            "n_docs": len(documents),
            "query_tokens_original": len(_encode_no_special(query)),
            "query_tokens_used": len(query_ids),
            "query_truncated": bool(query_was_truncated),
            "doc_tokens_original_sum": int(sum(original_lengths)),
            "doc_tokens_after_per_doc_cap_sum": int(sum(
                min(length, JINA_MAX_DOC_TOKENS_PER_DOC)
                for length in original_lengths
            )),
            "doc_tokens_used_sum": int(sum(used_lengths)),
            "docs_truncated": int(sum(
                used < original for used, original in zip(used_lengths, original_lengths)
            )),
            "max_doc_tokens_used": int(max(used_lengths, default=0)),
        }

        assert stats["doc_tokens_used_sum"] <= JINA_TOTAL_DOC_TOKEN_BUDGET
        assert stats["max_doc_tokens_used"] <= JINA_MAX_DOC_TOKENS_PER_DOC
        assert stats["query_tokens_used"] <= JINA_MAX_QUERY_TOKENS
        return prepared_query, prepared_documents, stats

    def jina_rerank_one(query, documents):
        prepared_query, prepared_documents, input_stats = jina_prepare_inputs(
            query, documents
        )

        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats(0)
        with torch.inference_mode():
            results = model.rerank(
                query=prepared_query,
                documents=prepared_documents,
                top_n=len(prepared_documents),
                return_embeddings=False,
            )
        torch.cuda.synchronize()

        scores_by_input_index = np.full(len(prepared_documents), np.nan, dtype=np.float32)
        preview = []
        for rank, item in enumerate(results):
            idx = int(item["index"])
            score = float(item["relevance_score"])
            if idx < 0 or idx >= len(prepared_documents):
                raise RuntimeError(f"Jina вернула некорректный index={idx}")
            scores_by_input_index[idx] = score
            if rank < 3:
                preview.append({
                    "rank": rank + 1,
                    "score": score,
                    "input_index": idx,
                    "document": str(item["document"])[:180],
                })

        if not np.isfinite(scores_by_input_index).all():
            raise RuntimeError("Jina не вернула конечную оценку для каждого документа.")

        input_stats["peak_allocated_gb"] = float(
            torch.cuda.max_memory_allocated(0) / 2**30
        )
        input_stats["peak_reserved_gb"] = float(
            torch.cuda.max_memory_reserved(0) / 2**30
        )
        input_stats["free_after_gb"] = float(
            torch.cuda.mem_get_info(0)[0] / 2**30
        )

        del results, prepared_documents
        return scores_by_input_index, preview, input_stats

    audit_task_indices = np.linspace(
        0, len(jina_tasks) - 1, num=min(150, len(jina_tasks)), dtype=int
    )
    audit_rows = []
    for task_idx in audit_task_indices:
        task = jina_tasks[int(task_idx)]
        _, _, stats = jina_prepare_inputs(task["query"], task["documents"])
        audit_rows.append({
            "split": task["split"],
            "query_id": task["query_id"],
            **stats,
        })
    token_budget_audit = pd.DataFrame(audit_rows)
    display(token_budget_audit.describe(include="all").T)
    token_budget_audit.to_csv(
        OUTPUT_DIR / "exp13_4_jina_token_budget_audit.csv", index=False
    )
    print("Token budget audit saved.")

    if JINA_MEMORY_PREFLIGHT and JINA_DONE_TASKS == 0:
        preflight_order = sorted(
            range(len(jina_tasks)),
            key=lambda i: (
                len(jina_tasks[i]["documents"]),
                sum(len(doc) for doc in jina_tasks[i]["documents"]),
            ),
            reverse=True,
        )[:min(JINA_PREFLIGHT_NUM_TASKS, len(jina_tasks))]

        preflight_peaks = []
        print(
            f"Memory stress preflight: testing {len(preflight_order)} "
            "largest query lists before full inference."
        )

        for preflight_num, preflight_idx in enumerate(preflight_order, 1):
            preflight_task = jina_tasks[preflight_idx]
            print(
                f"Preflight {preflight_num}/{len(preflight_order)} | "
                f"query_id={preflight_task['query_id']} | "
                f"documents={len(preflight_task['documents'])} | "
                f"raw_chars={sum(len(x) for x in preflight_task['documents'])}"
            )
            try:
                preflight_scores, preflight_preview, preflight_stats = jina_rerank_one(
                    preflight_task["query"], preflight_task["documents"]
                )
                peak_alloc = float(preflight_stats["peak_allocated_gb"])
                peak_reserved = float(preflight_stats["peak_reserved_gb"])
                preflight_peaks.append(peak_alloc)

                print(
                    f"Preflight passed | peak={peak_alloc:.2f} GiB | "
                    f"docs_tokens={preflight_stats['doc_tokens_used_sum']:,} | "
                    f"query_tokens={preflight_stats['query_tokens_used']:,} | "
                    f"free_after={preflight_stats['free_after_gb']:.2f} GiB"
                )

                del preflight_scores, preflight_preview, preflight_stats
                before_cleanup = jina_gpu_memory_snapshot()
                after_cleanup = jina_cleanup_memory()
                jina_record_memory(
                    "preflight_cleanup",
                    JINA_DONE_TASKS,
                    before=before_cleanup,
                    after=after_cleanup,
                    query_peak_allocated_gb=peak_alloc,
                    query_peak_reserved_gb=peak_reserved,
                    note=f"Stress preflight {preflight_num}/{len(preflight_order)}.",
                )

                if peak_alloc >= JINA_PREFLIGHT_STOP_AT_GB:
                    raise RuntimeError(
                        f"Preflight достиг {peak_alloc:.2f} GiB, порог "
                        f"{JINA_PREFLIGHT_STOP_AT_GB:.2f} GiB. Полный инференс "
                        "остановлен до начала расчёта. Не увеличивай лимиты; "
                        "следующий безопасный шаг — уменьшить общий бюджет "
                        "до 9 216 токенов и лимит документа до 384."
                    )

            except torch.cuda.OutOfMemoryError as err:
                try:
                    gc.collect()
                    torch.cuda.empty_cache()
                    torch.cuda.synchronize(0)
                except Exception:
                    pass
                raise RuntimeError(
                    "Memory stress preflight завершился CUDA OOM. Полный "
                    "инференс не запускался. Следующий тест: "
                    "JINA_TOTAL_DOC_TOKEN_BUDGET=9216, "
                    "JINA_MAX_DOC_TOKENS_PER_DOC=384, "
                    "JINA_MAX_QUERY_TOKENS=192."
                ) from err

        print(
            "Memory stress preflight complete | "
            f"max tested peak={max(preflight_peaks, default=float('nan')):.2f} GiB | "
            f"stop threshold={JINA_PREFLIGHT_STOP_AT_GB:.2f} GiB"
        )

    if JINA_DONE_TASKS == 0:
        task = jina_tasks[0]
        smoke_scores, smoke_preview, smoke_stats = jina_rerank_one(
            task["query"], task["documents"]
        )
        JINA_SCORES_FLAT[task["start"]:task["end"]] = smoke_scores
        JINA_DONE_TASKS = 1
        jina_save_checkpoint(JINA_CHECKPOINT_FILE, JINA_SCORES_FLAT, JINA_DONE_TASKS)

        print("Memory-safe smoke test passed.")
        print("Documents scored:", len(smoke_scores))
        print("Input token stats:", json.dumps(smoke_stats, ensure_ascii=False))
        print("Top-3 Jina results:")
        for item in smoke_preview:
            print(
                f"rank={item['rank']} | score={item['score']:.5f} | "
                f"input_index={item['input_index']} | {item['document']}"
            )
        smoke_peak_alloc = float(smoke_stats["peak_allocated_gb"])
        smoke_peak_reserved = float(smoke_stats["peak_reserved_gb"])
        del smoke_scores, smoke_preview, smoke_stats
        before_cleanup = jina_gpu_memory_snapshot()
        after_cleanup = jina_cleanup_memory()
        jina_record_memory(
            "smoke_test_cleanup",
            JINA_DONE_TASKS,
            before=before_cleanup,
            after=after_cleanup,
            query_peak_allocated_gb=smoke_peak_alloc,
            query_peak_reserved_gb=smoke_peak_reserved,
            note="Cleanup after the first real query smoke test.",
        )
    else:
        print("Resuming from task", JINA_DONE_TASKS + 1)

    start_task = JINA_DONE_TASKS
    session_start = time.monotonic()
    max_seconds = (
        None if JINA_MAX_SESSION_HOURS is None
        else JINA_MAX_SESSION_HOURS * 3600
    )
    last_saved_task = JINA_DONE_TASKS
    last_cleanup_task = JINA_DONE_TASKS
    recent_memory_rows = []

    for task_idx in range(start_task, JINA_NUM_TASKS):
        if max_seconds is not None and time.monotonic() - session_start >= max_seconds:
            jina_save_checkpoint(
                JINA_CHECKPOINT_FILE, JINA_SCORES_FLAT, task_idx
            )
            JINA_DONE_TASKS = task_idx
            before_cleanup = jina_gpu_memory_snapshot()
            after_cleanup = jina_cleanup_memory()
            jina_record_memory(
                "session_limit_cleanup",
                JINA_DONE_TASKS,
                before=before_cleanup,
                after=after_cleanup,
                note="Session time budget reached; checkpoint saved first.",
            )
            print("Session time budget reached; checkpoint saved and memory cleaned.")
            break

        task = jina_tasks[task_idx]
        try:
            scores_by_input_index, preview, stats = jina_rerank_one(
                task["query"], task["documents"]
            )
            JINA_SCORES_FLAT[task["start"]:task["end"]] = scores_by_input_index
            JINA_DONE_TASKS = task_idx + 1
            recent_memory_rows.append(dict(stats))

            query_peak_alloc = float(stats["peak_allocated_gb"])
            query_peak_reserved = float(stats["peak_reserved_gb"])
            del scores_by_input_index, preview, stats

            checkpoint_due = (
                JINA_DONE_TASKS - last_saved_task >= JINA_CHECKPOINT_EVERY_QUERIES
                or JINA_DONE_TASKS == JINA_NUM_TASKS
            )
            cleanup_due = (
                JINA_MEMORY_CLEANUP_ENABLED
                and JINA_DONE_TASKS - last_cleanup_task >= JINA_MEMORY_CLEANUP_EVERY_QUERIES
            )

            if checkpoint_due:
                jina_save_checkpoint(
                    JINA_CHECKPOINT_FILE, JINA_SCORES_FLAT, JINA_DONE_TASKS
                )
                last_saved_task = JINA_DONE_TASKS

            free_now_gb = float(torch.cuda.mem_get_info(0)[0] / 2**30)
            emergency_cleanup_due = (
                JINA_MEMORY_CLEANUP_ENABLED
                and free_now_gb < JINA_MEMORY_WARN_FREE_GB
            )

            if (emergency_cleanup_due or cleanup_due) and not checkpoint_due:
                jina_save_checkpoint(
                    JINA_CHECKPOINT_FILE, JINA_SCORES_FLAT, JINA_DONE_TASKS
                )
                last_saved_task = JINA_DONE_TASKS
                checkpoint_due = True

            after_cleanup = None
            if cleanup_due or emergency_cleanup_due or checkpoint_due:
                before_cleanup = jina_gpu_memory_snapshot()
                after_cleanup = jina_cleanup_memory()
                event_name = (
                    "emergency_cleanup" if emergency_cleanup_due
                    else "periodic_checkpoint_cleanup"
                )
                jina_record_memory(
                    event_name,
                    JINA_DONE_TASKS,
                    before=before_cleanup,
                    after=after_cleanup,
                    query_peak_allocated_gb=query_peak_alloc,
                    query_peak_reserved_gb=query_peak_reserved,
                    note=(
                        "Checkpoint saved before cleanup. "
                        + (
                            "Triggered by low free VRAM."
                            if emergency_cleanup_due else
                            "Scheduled periodic cleanup."
                        )
                    ),
                )
                last_cleanup_task = JINA_DONE_TASKS

            if checkpoint_due:
                elapsed = max(time.monotonic() - session_start, 1.0)
                done_now = JINA_DONE_TASKS - start_task
                qps = done_now / elapsed
                docs_done = sum(
                    len(t["row_indices"])
                    for t in jina_tasks[start_task:JINA_DONE_TASKS]
                )
                dps = docs_done / elapsed
                recent_df = pd.DataFrame(
                    recent_memory_rows[-JINA_CHECKPOINT_EVERY_QUERIES:]
                )
                peak_gb = (
                    float(recent_df["peak_allocated_gb"].max())
                    if not recent_df.empty else float("nan")
                )
                remaining = JINA_NUM_TASKS - JINA_DONE_TASKS
                eta_h = remaining / max(qps, 1e-9) / 3600
                post_free = (
                    after_cleanup["free_gb"] if after_cleanup else float("nan")
                )
                print(
                    f"Jina: {JINA_DONE_TASKS:,}/{JINA_NUM_TASKS:,} queries | "
                    f"{qps:.2f} queries/sec | {dps:.1f} docs/sec | "
                    f"batch peak {peak_gb:.2f} GiB | "
                    f"post-cleanup free {post_free:.2f} GiB | "
                    f"ETA {eta_h:.2f} h | checkpoint + cleanup saved"
                )
                recent_memory_rows = []

        except torch.cuda.OutOfMemoryError as err:
            jina_save_checkpoint(
                JINA_CHECKPOINT_FILE, JINA_SCORES_FLAT, JINA_DONE_TASKS
            )
            try:
                before_cleanup = jina_gpu_memory_snapshot()
                after_cleanup = jina_cleanup_memory()
                jina_record_memory(
                    "oom_recovery_cleanup",
                    JINA_DONE_TASKS,
                    before=before_cleanup,
                    after=after_cleanup,
                    note="CUDA OOM on current query; completed queries checkpointed.",
                )
            except Exception as cleanup_exc:
                print("Post-OOM cleanup reported:", repr(cleanup_exc))
            raise RuntimeError(
                "CUDA OOM на одном списке документов. Все завершённые запросы "
                "сохранены в checkpoint. Очистка памяти выполнена; не смешивай "
                "результаты с другим токенным бюджетом. Следующий тестовый "
                "режим: JINA_TOTAL_DOC_TOKEN_BUDGET=9216, "
                "JINA_MAX_DOC_TOKENS_PER_DOC=384, "
                "JINA_MAX_QUERY_TOKENS=192; нужен новый fingerprint."
            ) from err
        except RuntimeError as err:
            if "out of memory" in str(err).lower():
                jina_save_checkpoint(
                    JINA_CHECKPOINT_FILE, JINA_SCORES_FLAT, JINA_DONE_TASKS
                )
                try:
                    before_cleanup = jina_gpu_memory_snapshot()
                    after_cleanup = jina_cleanup_memory()
                    jina_record_memory(
                        "oom_recovery_cleanup",
                        JINA_DONE_TASKS,
                        before=before_cleanup,
                        after=after_cleanup,
                        note="RuntimeError OOM; completed queries checkpointed.",
                    )
                except Exception as cleanup_exc:
                    print("Post-OOM cleanup reported:", repr(cleanup_exc))
                raise RuntimeError(
                    "CUDA OOM. Завершённые запросы сохранены; выполнена попытка "
                    "очистки памяти. Следующий тестовый режим: "
                    "document=384, total_doc_budget=9216, query=192; "
                    "измени fingerprint."
                ) from err
            raise

    JINA_COMPLETE = JINA_DONE_TASKS >= JINA_NUM_TASKS
    jina_save_checkpoint(
        JINA_COMPLETE_FILE if JINA_COMPLETE else JINA_CHECKPOINT_FILE,
        JINA_SCORES_FLAT,
        JINA_DONE_TASKS,
    )

    del model, jina_tokenizer
    final_before_cleanup = jina_gpu_memory_snapshot()
    final_after_cleanup = jina_cleanup_memory()
    jina_record_memory(
        "model_unloaded_final_cleanup",
        JINA_DONE_TASKS,
        before=final_before_cleanup,
        after=final_after_cleanup,
        note="Jina model and tokenizer deleted after inference.",
    )
    print("Memory telemetry saved:", JINA_MEMORY_TELEMETRY_FILE)

if JINA_COMPLETE:
    print("Jina scoring complete:", JINA_NUM_TASKS, "query lists.")
else:
    print(
        "Jina scoring is incomplete. Увеличь доступное время GPU и повтори запуск."
    )

### Rich100

In [ ]:
if JINA_COMPLETE:
    train_jina = np.full(len(train_fe), np.nan, dtype=np.float32)
    test_jina = np.full(len(test_fe), np.nan, dtype=np.float32)

    for task in jina_tasks:
        task_scores = JINA_SCORES_FLAT[task["start"]:task["end"]]
        assert len(task_scores) == len(task["row_indices"])
        if task["split"] == "train":
            train_jina[task["row_indices"]] = task_scores
        else:
            test_jina[task["row_indices"]] = task_scores

    train_fe["jina_v35_reranker_score"] = train_jina
    test_fe["jina_v35_reranker_score"] = test_jina

    # Percentile rank within query; rows outside Top-30 remain NaN.
    train_fe["jina_v35_query_rank"] = (
        train_fe.groupby("query_id", sort=False)["jina_v35_reranker_score"]
        .rank(pct=True, method="average").astype(np.float32)
    )
    test_fe["jina_v35_query_rank"] = (
        test_fe.groupby("query_id", sort=False)["jina_v35_reranker_score"]
        .rank(pct=True, method="average").astype(np.float32)
    )

    JINA_NEW_COLS = [
        "jina_v35_reranker_score",
        "jina_v35_query_rank",
    ]
    RICH100_FEATURES = RICH_FEATURES + JINA_NEW_COLS
    assert len(RICH_FEATURES) == 98 and len(RICH100_FEATURES) == 100
    assert len(set(RICH100_FEATURES)) == 100

    train_fe[["id", "query_id", "document_id", "article_id", "relevance"] + RICH100_FEATURES].to_csv(
        OUTPUT_DIR / "train_features_exp13_4_jina_rich100.csv",
        index=False,
        float_format="%.9g",
    )
    test_fe[["id", "query_id", "document_id", "article_id"] + RICH100_FEATURES].to_csv(
        OUTPUT_DIR / "test_features_exp13_4_jina_rich100.csv",
        index=False,
        float_format="%.9g",
    )

    print("Rich98 baseline:", len(RICH_FEATURES))
    print("Rich100 with Jina:", len(RICH100_FEATURES))
    print("Jina scored rows — train:", int(train_fe.jina_v35_reranker_score.notna().sum()),
          "| test:", int(test_fe.jina_v35_reranker_score.notna().sum()))
    display(train_fe[["id", "query_id", "qwen_reranker_score",
                      "jina_v35_reranker_score", "jina_v35_query_rank"]].head(8))
else:
    print("Skip feature assembly and model training until Jina scoring is complete.")

### Пять групповых фолдов

In [ ]:
if JINA_COMPLETE:
    from sklearn.model_selection import GroupKFold

    class UnionFind:
        def __init__(self):
            self.parent = {}
            self.size = {}

        def find(self, x):
            if x not in self.parent:
                self.parent[x] = x
                self.size[x] = 1
            if self.parent[x] != x:
                self.parent[x] = self.find(self.parent[x])
            return self.parent[x]

        def union(self, a, b):
            pa, pb = self.find(a), self.find(b)
            if pa == pb:
                return
            if self.size[pa] < self.size[pb]:
                pa, pb = pb, pa
            self.parent[pb] = pa
            self.size[pa] += self.size[pb]

    uf = UnionFind()
    for q, d in zip(
        train_fe["query_id"].astype(str),
        train_fe["document_id"].astype(str),
    ):
        uf.union("q:" + q, "d:" + d)

    roots = [
        uf.find("q:" + q)
        for q in train_fe["query_id"].astype(str)
    ]
    component_ids = pd.factorize(np.asarray(roots, dtype=object), sort=False)[0]
    labels = train_fe["relevance"].to_numpy(dtype=np.int32)

    SPLITS = list(
        GroupKFold(n_splits=N_SPLITS).split(train_fe, labels, groups=component_ids)
    )

    fold_rows = []
    fold_id = np.full(len(train_fe), -1, dtype=np.int8)
    for fold, (tr_idx, va_idx) in enumerate(SPLITS, 1):
        trf = train_fe.iloc[tr_idx]
        vaf = train_fe.iloc[va_idx]
        q_overlap = len(set(trf.query_id.astype(str)) & set(vaf.query_id.astype(str)))
        d_overlap = len(set(trf.document_id.astype(str)) & set(vaf.document_id.astype(str)))
        assert q_overlap == 0 and d_overlap == 0
        fold_id[va_idx] = fold
        fold_rows.append({
            "fold": fold,
            "train_rows": len(tr_idx),
            "valid_rows": len(va_idx),
            "valid_queries": vaf.query_id.nunique(),
            "valid_positive_rate": float(vaf.relevance.mean()),
            "query_overlap": q_overlap,
            "document_overlap": d_overlap,
        })

    assert (fold_id > 0).all()
    fold_audit = pd.DataFrame(fold_rows)
    display(fold_audit.round(5))
    print("Connected components:", len(np.unique(component_ids)))
    print("Largest component row share:",
          round(pd.Series(component_ids).value_counts().iloc[0] / len(train_fe), 4))
    fold_audit.to_csv(OUTPUT_DIR / "exp13_4_jina_cv_integrity.csv", index=False)
else:
    print("CV setup skipped: Jina inference is not complete.")

### Проверка Rich100

In [ ]:
if JINA_COMPLETE:
    import lightgbm as lgb

    def ndcg10(frame, raw_scores):
        tmp = pd.DataFrame({
            "q": frame["query_id"].astype(str).to_numpy(),
            "y": frame["relevance"].to_numpy(dtype=np.float64),
            "s": np.asarray(raw_scores, dtype=np.float64),
        })
        values = []
        for _, group in tmp.groupby("q", sort=False):
            y = group["y"].to_numpy()
            pred = np.nan_to_num(
                group["s"].to_numpy(),
                nan=-1e30, posinf=1e30, neginf=-1e30,
            )
            order = np.argsort(-pred, kind="mergesort")[:10]
            ideal = np.sort(y)[::-1][:10]
            discounts = np.log2(np.arange(2, 2 + len(order)))
            ideal_discounts = np.log2(np.arange(2, 2 + len(ideal)))
            dcg = np.sum((2 ** y[order] - 1) / discounts)
            idcg = np.sum((2 ** ideal - 1) / ideal_discounts)
            values.append(dcg / idcg if idcg > 0 else 0.0)
        return float(np.mean(values))

    MODEL_SPECS = {
        "rich100_jina": list(RICH100_FEATURES),
    }
    CAT_COLS = [c for c in ["ingest_source", "query_question_word"] if c in RICH100_FEATURES]
    JINA_MISSING_COLS = set(JINA_NEW_COLS)

    MTRAIN = train_fe[RICH100_FEATURES].copy()
    MTEST = test_fe[RICH100_FEATURES].copy()

    NUM_COLS = [c for c in RICH100_FEATURES if c not in CAT_COLS]
    for col in NUM_COLS:
        MTRAIN[col] = pd.to_numeric(MTRAIN[col], errors="coerce").replace(
            [np.inf, -np.inf], np.nan
        ).astype(np.float32)
        MTEST[col] = pd.to_numeric(MTEST[col], errors="coerce").replace(
            [np.inf, -np.inf], np.nan
        ).astype(np.float32)

    for col in CAT_COLS:
        a = MTRAIN[col].fillna("__MISSING__").astype(str)
        b = MTEST[col].fillna("__MISSING__").astype(str)
        categories = pd.Index(pd.concat([a, b], ignore_index=True).unique())
        MTRAIN[col] = pd.Categorical(a, categories=categories)
        MTEST[col] = pd.Categorical(b, categories=categories)

    def query_sort(indices):
        indices = np.asarray(indices)
        order = np.argsort(
            train_fe.iloc[indices]["query_id"].astype(str).to_numpy(),
            kind="mergesort",
        )
        sorted_indices = indices[order]
        group_sizes = train_fe.iloc[sorted_indices].groupby(
            "query_id", sort=False
        ).size().tolist()
        return sorted_indices, group_sizes

    def fold_matrices(train_indices, valid_indices, features):
        X_train = MTRAIN.iloc[train_indices][features].copy()
        X_valid = MTRAIN.iloc[valid_indices][features].copy()
        for col in features:
            if col in CAT_COLS or col in JINA_MISSING_COLS:
                continue
            median = float(X_train[col].median()) if X_train[col].notna().any() else 0.0
            if not np.isfinite(median):
                median = 0.0
            X_train[col] = X_train[col].fillna(median).astype(np.float32)
            X_valid[col] = X_valid[col].fillna(median).astype(np.float32)
        # NaN в Jina score/rank намеренно сохраняются: это строки вне Top-30, LightGBM обрабатывает missing values самостоятельно.
        return X_train, X_valid

    def full_matrices(features):
        X_train = MTRAIN[features].copy()
        X_test = MTEST[features].copy()
        for col in features:
            if col in CAT_COLS or col in JINA_MISSING_COLS:
                continue
            median = float(X_train[col].median()) if X_train[col].notna().any() else 0.0
            if not np.isfinite(median):
                median = 0.0
            X_train[col] = X_train[col].fillna(median).astype(np.float32)
            X_test[col] = X_test[col].fillna(median).astype(np.float32)
        return X_train, X_test

    def make_ranker(seed, n_estimators=N_ESTIMATORS):
        return lgb.LGBMRanker(
            objective="lambdarank",
            metric="ndcg",
            n_estimators=int(n_estimators),
            learning_rate=RANKER_PARAMS["learning_rate"],
            num_leaves=RANKER_PARAMS["num_leaves"],
            min_child_samples=RANKER_PARAMS["min_child_samples"],
            max_depth=-1,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=RANKER_PARAMS["reg_lambda"],
            label_gain=[0, 1],
            random_state=int(seed),
            n_jobs=-1,
            verbosity=-1,
        )

    def run_fold(features, train_indices, valid_indices, fold_seed):
        train_sorted, train_group_sizes = query_sort(train_indices)
        valid_sorted, valid_group_sizes = query_sort(valid_indices)

        X_train, X_valid_sorted = fold_matrices(train_sorted, valid_sorted, features)
        model = make_ranker(fold_seed)
        model.fit(
            X_train,
            labels[train_sorted],
            group=train_group_sizes,
            eval_set=[(X_valid_sorted, labels[valid_sorted])],
            eval_group=[valid_group_sizes],
            eval_at=[10],
            categorical_feature=[c for c in CAT_COLS if c in features],
            callbacks=[
                lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
                lgb.log_evaluation(0),
            ],
        )

        _, X_valid_original_order = fold_matrices(train_sorted, valid_indices, features)
        best_iteration = int(model.best_iteration_ or N_ESTIMATORS)
        predictions = np.asarray(
            model.predict(X_valid_original_order, num_iteration=best_iteration),
            dtype=np.float32,
        )
        score = ndcg10(train_fe.iloc[valid_indices], predictions)
        return model, predictions, best_iteration, score

    OOF = {
        name: np.full(len(train_fe), np.nan, dtype=np.float32)
        for name in MODEL_SPECS
    }
    ITERATIONS = {name: [] for name in MODEL_SPECS}
    CV_ROWS = []

    for name, features in MODEL_SPECS.items():
        print("\n" + "=" * 65)
        print(name, "| features:", len(features))
        for fold, (tr_idx, va_idx) in enumerate(SPLITS, 1):
            model, pred, best_it, score = run_fold(
                features, tr_idx, va_idx, RANDOM_STATE + fold
            )
            OOF[name][va_idx] = pred
            ITERATIONS[name].append(best_it)
            CV_ROWS.append({
                "model": name,
                "fold": fold,
                "ndcg10": score,
                "best_iteration": best_it,
                "valid_rows": len(va_idx),
                "valid_queries": train_fe.iloc[va_idx].query_id.nunique(),
            })

            np.save(OUTPUT_DIR / f"exp13_4_jina_oof_{name}.npy", OOF[name])
            pd.DataFrame(CV_ROWS).to_csv(
                OUTPUT_DIR / "exp13_4_jina_cv_partial.csv", index=False
            )
            print(
                f"Fold {fold}/{N_SPLITS}: NDCG@10={score:.6f} "
                f"| best_iteration={best_it}"
            )
            del model, pred
            gc.collect()

        assert np.isfinite(OOF[name]).all()
        print("OOF NDCG@10:", name, ndcg10(train_fe, OOF[name]))

    cv_folds = pd.DataFrame(CV_ROWS)
    cv_summary = cv_folds.groupby("model", as_index=False).agg(
        fold_mean=("ndcg10", "mean"),
        fold_std=("ndcg10", "std"),
        median_iteration=("best_iteration", "median"),
    )
    cv_summary["OOF_NDCG@10"] = [
        ndcg10(train_fe, OOF[name]) for name in cv_summary["model"]
    ]
    cv_summary = cv_summary.sort_values(
        "OOF_NDCG@10", ascending=False
    ).reset_index(drop=True)

    cv_folds.to_csv(OUTPUT_DIR / "exp13_4_jina_cv_folds.csv", index=False)
    cv_summary.to_csv(OUTPUT_DIR / "exp13_4_jina_cv_summary.csv", index=False)

    display(cv_summary.round(6))
else:
    print("CV skipped: Jina inference is not complete.")

### Обучение полной модели Rich100

In [ ]:
if JINA_COMPLETE:
    FINAL_PRED = {}
    FINAL_ITERS = {}
    full_sorted, full_group_sizes = query_sort(np.arange(len(train_fe)))

    def query_rank(frame, scores):
        values = pd.Series(np.asarray(scores, dtype=np.float64), index=np.arange(len(frame)))
        groups = frame["query_id"].astype(str).reset_index(drop=True)
        return values.groupby(groups).rank(method="average", pct=True).to_numpy(dtype=np.float32)

    def create_submission(scores, filename):
        scores = np.asarray(scores, dtype=np.float32)
        assert len(scores) == len(test_fe) and np.isfinite(scores).all()

        lookup = pd.Series(scores, index=test_fe["id"].astype(str))
        ids = (
            sample_submission["id"]
            if sample_submission is not None
            else test_fe["id"]
        ).astype(str).reset_index(drop=True)

        submission = pd.DataFrame({
            "id": ids,
            "prediction": ids.map(lookup),
        })
        assert submission["id"].is_unique
        assert set(submission["id"]) == set(test_fe["id"].astype(str))
        assert submission["prediction"].notna().all()
        assert np.isfinite(submission["prediction"]).all()

        submission.to_csv(OUTPUT_DIR / filename, index=False)
        print("Saved:", filename, submission.shape)
        return submission

    for name, features in MODEL_SPECS.items():
        iterations = max(60, int(np.median(ITERATIONS[name])))
        X_train, X_test = full_matrices(features)
        X_train = X_train.iloc[full_sorted].reset_index(drop=True)

        model = make_ranker(RANDOM_STATE, n_estimators=iterations)
        model.fit(
            X_train,
            labels[full_sorted],
            group=full_group_sizes,
            categorical_feature=[c for c in CAT_COLS if c in features],
        )

        raw_pred = np.asarray(model.predict(X_test), dtype=np.float32)
        assert len(raw_pred) == len(test_fe) and np.isfinite(raw_pred).all()
        FINAL_PRED[name] = query_rank(test_fe, raw_pred)
        FINAL_ITERS[name] = iterations

        model.booster_.save_model(str(OUTPUT_DIR / f"exp13_4_jina_model_{name}.txt"))
        importance = pd.DataFrame({
            "feature": features,
            "gain": model.booster_.feature_importance(importance_type="gain"),
        }).sort_values("gain", ascending=False)
        importance.to_csv(
            OUTPUT_DIR / f"exp13_4_jina_importance_{name}.csv", index=False
        )

        filename = "submission.csv"
        create_submission(FINAL_PRED[name], filename)
        create_submission(FINAL_PRED[name], "submission_exp13_4_jina_rich100.csv")

        del model, X_train, X_test, raw_pred
        gc.collect()

    print("Final Rich100 submission created.")
else:
    print("Final models skipped: Jina inference is incomplete or RUN_FINAL_MODELS=False.")

In [ ]:
CHAMPION13_OOF = OOF["rich100_jina"].copy()
CHAMPION13_TEST = FINAL_PRED["rich100_jina"].copy()
assert np.isfinite(CHAMPION13_OOF).all()
assert np.isfinite(CHAMPION13_TEST).all()
print("Rich100 готов:", len(RICH100_FEATURES), "признаков")
print("Experiment 13.4 OOF NDCG@10:", round(ndcg10(train_fe, CHAMPION13_OOF), 6))

## 5. Experiment 18 — ColBERT v2

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 5
MODEL_ID = "jinaai/jina-colbert-v2"
EMBEDDING_DIM = 128
DOC_MAX_LENGTH = 512
QUERY_MAX_LENGTH = 32
QUERY_BATCH_SIZE = 16
DOC_BATCH_SIZE = 8
QUERY_CHUNK_SIZE = 256
DOC_CHUNK_SIZE = 128
RUN_FULL_TEXT = True
RUN_BODY_TEXT = True
RUN_TITLE_TEXT = True
CACHE_ROOT = OUTPUT_DIR / "embedding_cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

import lightgbm as lgb
from sentence_transformers import MultiVectorEncoder

raw_train = train.copy()
raw_test = test.copy()
RAW_TRAIN_REQUIRED = set(REQUIRED_TRAIN)
RAW_TEST_REQUIRED = set(REQUIRED_TEST)
print("Model:", MODEL_ID, "| dimension:", EMBEDDING_DIM)


In [ ]:
raw_train["id"] = raw_train["id"].astype(str)
raw_test["id"] = raw_test["id"].astype(str)
train_fe["id"] = train_fe["id"].astype(str)
test_fe["id"] = test_fe["id"].astype(str)

def align_raw(raw, features, split_name):
    if not raw["id"].is_unique or not features["id"].is_unique:
        raise ValueError(f"{split_name}: повторяющиеся id.")
    if set(raw["id"]) != set(features["id"]):
        raise ValueError(f"{split_name}: raw и Rich100 содержат разные id.")
    aligned = raw.set_index("id", drop=False).loc[features["id"].tolist()].reset_index(drop=True)
    for col in ["query_id", "document_id"]:
        if not aligned[col].astype(str).reset_index(drop=True).equals(features[col].astype(str).reset_index(drop=True)):
            raise ValueError(f"{split_name}: {col} не совпадает между raw и Rich100.")
    if split_name == "train":
        if not np.array_equal(aligned["relevance"].to_numpy(), features["relevance"].to_numpy()):
            raise ValueError("Train labels в raw и Rich100 не совпадают.")
    return aligned

raw_train = align_raw(raw_train, train_fe, "train")
raw_test = align_raw(raw_test, test_fe, "test")
assert len(RICH100_FEATURES) == 100
assert set(RICH100_FEATURES).issubset(train_fe.columns)
assert set(RICH100_FEATURES).issubset(test_fe.columns)
assert set(RAW_TRAIN_REQUIRED).issubset(raw_train.columns)
assert set(RAW_TEST_REQUIRED).issubset(raw_test.columns)
print("Rich100:", len(RICH100_FEATURES), "признаков")


In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value)
    value = value.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value

def make_doc_variants(frame):
    title = frame["title"].map(clean_text)
    body = frame["text"].map(clean_text)

    full = [
        (f"{t}\n{b}".strip() if t and b else (t or b)) or "[EMPTY]"
        for t, b in zip(title, body)
    ]
    body_only = [x if x else "[EMPTY]" for x in body]
    title_only = [x if x else "[EMPTY]" for x in title]

    return {
        "full": full,
        "text": body_only,
        "title": title_only,
    }

all_raw = pd.concat([
    raw_train.assign(_split="train"),
    raw_test.assign(_split="test"),
], ignore_index=True)

all_raw["query_id"] = all_raw["query_id"].astype(str)
all_raw["query_clean"] = all_raw["query"].map(clean_text)

query_check = all_raw.groupby(["_split", "query_id"])["query_clean"].nunique()
if (query_check > 1).any():
    bad = query_check[query_check > 1].index[:5].tolist()
    raise ValueError(f"Один query_id связан с разными текстами запроса: {bad}")

query_table = (
    all_raw[["_split", "query_id", "query_clean"]]
    .drop_duplicates(["_split", "query_id"])
    .reset_index(drop=True)
)

def text_key(text):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

query_table["query_key"] = query_table["query_clean"].map(text_key)
query_key_by_id = {
    (str(split_name), str(query_id)): key
    for split_name, query_id, key in zip(
        query_table["_split"], query_table["query_id"], query_table["query_key"]
    )
}

variants = make_doc_variants(all_raw)
for name, texts in variants.items():
    all_raw[f"_doc_{name}"] = texts
    all_raw[f"_key_{name}"] = [text_key(x) for x in texts]

doc_text_by_key = {}
ACTIVE_CHANNELS = [
    name for name, enabled in [
        ("full", RUN_FULL_TEXT),
        ("text", RUN_BODY_TEXT),
        ("title", RUN_TITLE_TEXT),
    ] if enabled
]

for name in ACTIVE_CHANNELS:
    all_raw[f"_doc_{name}"] = variants[name]
    all_raw[f"_key_{name}"] = [text_key(x) for x in variants[name]]
    for key, text_value in zip(all_raw[f"_key_{name}"], all_raw[f"_doc_{name}"]):
        doc_text_by_key.setdefault(key, text_value)

query_text_by_key = dict(zip(query_table["query_key"], query_table["query_clean"]))

print("Unique queries:", len(query_text_by_key))
print("Unique document strings across all channels:", len(doc_text_by_key))
print("Missing query text:", int((all_raw["query_clean"] == "").sum()))
print("Missing title:", int(all_raw["title"].isna().sum()))
print("Missing passage text:", int(all_raw["text"].isna().sum()))
print("Empty full document:", int((all_raw["_doc_full"] == "[EMPTY]").sum()))


In [ ]:
gc.collect()
torch.cuda.empty_cache()

model = MultiVectorEncoder(
    MODEL_ID,
    trust_remote_code=True,
    device="cuda",
    truncate_dim=EMBEDDING_DIM,
)
model = model.to("cuda")
model = model.half()
model.eval()

transformer_module = model[0]
if hasattr(transformer_module, "document_length"):
    current_doc_length = transformer_module.document_length
    if current_doc_length is None or current_doc_length > DOC_MAX_LENGTH:
        transformer_module.document_length = DOC_MAX_LENGTH

if hasattr(transformer_module, "query_length"):
    current_query_length = transformer_module.query_length
    if current_query_length is None or current_query_length > QUERY_MAX_LENGTH:
        transformer_module.query_length = QUERY_MAX_LENGTH

# В этом эксперименте используем именно сумму MaxSim.
if getattr(model, "similarity_fn_name", None) != "maxsim":
    model.similarity_fn_name = "maxsim"

print(model)
print("Prompts:", getattr(model, "prompts", None))
print("Similarity function:", getattr(model, "similarity_fn_name", None))
print("Document length:", getattr(transformer_module, "document_length", None))
print("Query length:", getattr(transformer_module, "query_length", None))
print("Native embedding dimension:", model.get_embedding_dimension())
print("Experiment embedding dimension:", EMBEDDING_DIM)
print("Model default truncate_dim:", getattr(model, "truncate_dim", None))

# Проверяем, насколько часто документам потребуется усечение до 512 токенов.
tokenizer = model.tokenizer
audit_texts = list(doc_text_by_key.values())[:2000]
audit_lengths = [
    len(tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])
    for text in audit_texts
]
if audit_lengths:
    print(
        "Document token audit | p50:",
        int(np.percentile(audit_lengths, 50)),
        "| p95:",
        int(np.percentile(audit_lengths, 95)),
        "| max:",
        int(np.max(audit_lengths)),
        "| share > cap:",
        round(float(np.mean(np.asarray(audit_lengths) > DOC_MAX_LENGTH)), 4),
    )

free_gb, total_gb = torch.cuda.mem_get_info(0)
print(f"GPU free after load: {free_gb / 2**30:.2f} / {total_gb / 2**30:.2f} GiB")

sample_query = query_table.iloc[0]["query_clean"]
sample_docs = [
    all_raw.iloc[0]["_doc_full"],
    all_raw.iloc[min(1, len(all_raw)-1)]["_doc_full"],
]

with torch.inference_mode():
    q_smoke = model.encode_query(sample_query, batch_size=1, show_progress_bar=False)
    d_smoke = model.encode_document(sample_docs, batch_size=2, show_progress_bar=False)
    smoke_scores = model.similarity(q_smoke, d_smoke)

assert q_smoke.shape[-1] == EMBEDDING_DIM, f"Query dim={q_smoke.shape[-1]}, expected {EMBEDDING_DIM}"
assert d_smoke[0].shape[-1] == EMBEDDING_DIM, f"Document dim={d_smoke[0].shape[-1]}, expected {EMBEDDING_DIM}"
smoke_scores = torch.as_tensor(smoke_scores).float().cpu().numpy().reshape(-1)
assert len(smoke_scores) == 2 and np.isfinite(smoke_scores).all()
print("Smoke test passed:", np.round(smoke_scores, 5).tolist())

del q_smoke, d_smoke, smoke_scores
gc.collect()
torch.cuda.empty_cache()


In [ ]:
RESUME_CACHE_DIR = None

def fingerprint_keys(keys, extra):
    h = hashlib.sha256(extra.encode("utf-8"))
    for key in keys:
        h.update(key.encode("utf-8"))
    return h.hexdigest()[:16]

query_keys = list(query_text_by_key.keys())
doc_keys = list(doc_text_by_key.keys())

query_fp = fingerprint_keys(query_keys, f"{MODEL_ID}|dim={EMBEDDING_DIM}|query|{QUERY_MAX_LENGTH}")
doc_fp = fingerprint_keys(doc_keys, f"{MODEL_ID}|dim={EMBEDDING_DIM}|document|{DOC_MAX_LENGTH}")

QUERY_CACHE_NAME = f"queries_{query_fp}"
DOC_CACHE_NAME = f"documents_{doc_fp}"

QUERY_CACHE_WRITE = CACHE_ROOT / QUERY_CACHE_NAME
DOC_CACHE_WRITE = CACHE_ROOT / DOC_CACHE_NAME
QUERY_CACHE_WRITE.mkdir(parents=True, exist_ok=True)
DOC_CACHE_WRITE.mkdir(parents=True, exist_ok=True)

def find_resume_cache(cache_name, write_dir):
    if RESUME_CACHE_DIR:
        candidate = Path(RESUME_CACHE_DIR) / cache_name
        if candidate.exists():
            return candidate

    for root in [OUTPUT_DIR]:
        if not root.exists():
            continue
        try:
            matches = list(root.rglob(cache_name))
            matches = [p for p in matches if p.is_dir()]
            if matches:
                return sorted(matches, key=lambda p: len(str(p)))[0]
        except (PermissionError, OSError):
            pass
    return write_dir

QUERY_CACHE_READ = find_resume_cache(QUERY_CACHE_NAME, QUERY_CACHE_WRITE)
DOC_CACHE_READ = find_resume_cache(DOC_CACHE_NAME, DOC_CACHE_WRITE)

print("Query cache read :", QUERY_CACHE_READ)
print("Query cache write:", QUERY_CACHE_WRITE)
print("Doc cache read   :", DOC_CACHE_READ)
print("Doc cache write  :", DOC_CACHE_WRITE)


In [ ]:
def cache_files(read_dir, write_dir):
    paths = []
    seen = set()
    for folder in [read_dir, write_dir]:
        if folder.exists():
            for path in sorted(folder.glob("batch_*.npz")):
                key = str(path.resolve())
                if key not in seen:
                    seen.add(key)
                    paths.append(path)
    return paths

def load_cached_embeddings(read_dir, write_dir):
    store = {}
    backing_arrays = []

    for path in cache_files(read_dir, write_dir):
        with np.load(path, allow_pickle=False) as data:
            keys = data["keys"].astype(str)
            offsets = data["offsets"].astype(np.int64)
            values = data["values"]

            backing_arrays.append(values)
            for i, key in enumerate(keys):
                if key not in store:
                    store[key] = values[offsets[i]:offsets[i+1]]

    return store, backing_arrays

def encode_to_cache(text_by_key, task, read_dir, write_dir, outer_chunk_size, initial_batch_size):
    cached, _ = load_cached_embeddings(read_dir, write_dir)
    missing = [key for key in text_by_key if key not in cached]

    print(
        f"{task}: total={len(text_by_key):,} | cached={len(cached):,} "
        f"| remaining={len(missing):,}"
    )

    existing_paths = cache_files(read_dir, write_dir)
    next_batch_number = 0
    for path in existing_paths:
        match = re.search(r"batch_(\d+)\.npz$", path.name)
        if match:
            next_batch_number = max(next_batch_number, int(match.group(1)) + 1)

    batch_size = initial_batch_size
    started = time.monotonic()

    for start in range(0, len(missing), outer_chunk_size):
        batch_keys = missing[start:start + outer_chunk_size]
        batch_texts = [text_by_key[key] for key in batch_keys]

        while True:
            try:
                with torch.inference_mode():
                    if task == "query":
                        outputs = model.encode_query(
                            batch_texts,
                            batch_size=batch_size,
                            show_progress_bar=False,
                            convert_to_numpy=True,
                        )
                    else:
                        outputs = model.encode_document(
                            batch_texts,
                            batch_size=batch_size,
                            show_progress_bar=False,
                            convert_to_numpy=True,
                        )
                break
            except RuntimeError as exc:
                is_oom = "out of memory" in str(exc).lower()
                if not is_oom or batch_size <= 1:
                    raise
                print(f"{task}: CUDA OOM at batch_size={batch_size}; retry with {max(1, batch_size//2)}")
                batch_size = max(1, batch_size // 2)
                gc.collect()
                torch.cuda.empty_cache()

        arrays = [
            np.asarray(x, dtype=np.float16)
            for x in outputs
        ]
        lengths = np.asarray([len(x) for x in arrays], dtype=np.int32)
        offsets = np.concatenate([[0], np.cumsum(lengths)]).astype(np.int64)
        values = np.concatenate(arrays, axis=0) if arrays else np.empty((0, EMBEDDING_DIM), dtype=np.float16)

        out_path = write_dir / f"batch_{next_batch_number:06d}.npz"
        np.savez(
            out_path,
            keys=np.asarray(batch_keys, dtype="U40"),
            offsets=offsets,
            values=values,
        )
        next_batch_number += 1

        done = min(start + len(batch_keys), len(missing))
        elapsed = time.monotonic() - started
        rate = done / max(elapsed, 1e-6)
        eta = (len(missing) - done) / max(rate, 1e-6)

        print(
            f"{task}: {done:,}/{len(missing):,} new | "
            f"{rate:.1f} texts/sec | ETA {eta/60:.1f} min | "
            f"saved {out_path.name}"
        )

        del outputs, arrays, values, batch_texts
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    store, backing = load_cached_embeddings(read_dir, write_dir)
    missing_after = set(text_by_key) - set(store)
    if missing_after:
        raise RuntimeError(f"{task}: cache incomplete, missing {len(missing_after)} embeddings.")

    print(f"{task}: cache ready ({len(store):,} embeddings).")
    return store, backing


In [ ]:
query_embeddings, query_backing = encode_to_cache(
    query_text_by_key,
    task="query",
    read_dir=QUERY_CACHE_READ,
    write_dir=QUERY_CACHE_WRITE,
    outer_chunk_size=QUERY_CHUNK_SIZE,
    initial_batch_size=QUERY_BATCH_SIZE,
)

doc_embeddings, doc_backing = encode_to_cache(
    doc_text_by_key,
    task="document",
    read_dir=DOC_CACHE_READ,
    write_dir=DOC_CACHE_WRITE,
    outer_chunk_size=DOC_CHUNK_SIZE,
    initial_batch_size=DOC_BATCH_SIZE,
)

print("Query embeddings:", len(query_embeddings))
print("Document embeddings:", len(doc_embeddings))
print("Query cache files:", len(cache_files(QUERY_CACHE_READ, QUERY_CACHE_WRITE)))
print("Document cache files:", len(cache_files(DOC_CACHE_READ, DOC_CACHE_WRITE)))


In [ ]:
def score_query_group(split_name, query_id, group, channel):
    qkey = query_key_by_id[(str(split_name), str(query_id))]
    q_np = query_embeddings[qkey]
    q_tensor = torch.from_numpy(np.asarray(q_np)).to("cuda")

    keys = group[f"_key_{channel}"].tolist()
    d_tensors = [
        torch.from_numpy(np.asarray(doc_embeddings[key])).to("cuda")
        for key in keys
    ]

    with torch.inference_mode():
        scores = model.similarity(q_tensor, d_tensors)

    scores = torch.as_tensor(scores).float().reshape(-1).cpu().numpy()
    if len(scores) != len(group) or not np.isfinite(scores).all():
        raise RuntimeError(f"Некорректные MaxSim scores для query_id={query_id}")

    query_token_count = int(q_np.shape[0])
    return scores, query_token_count


score_rows = []
score_started = time.monotonic()

for channel in ["full", "text", "title"]:
    enabled = (
        (channel == "full" and RUN_FULL_TEXT)
        or (channel == "text" and RUN_BODY_TEXT)
        or (channel == "title" and RUN_TITLE_TEXT)
    )
    if not enabled:
        continue

    score_col = f"colbert_{channel}_maxsim"
    all_raw[score_col] = np.nan
    all_raw[f"colbert_{channel}_query_tokens"] = np.nan

    query_groups = list(all_raw.groupby(["_split", "query_id"], sort=False))
    for i, ((split_name, query_id), group) in enumerate(query_groups, 1):
        scores, q_tokens = score_query_group(str(split_name), str(query_id), group, channel)
        all_raw.loc[group.index, score_col] = scores
        all_raw.loc[group.index, f"colbert_{channel}_query_tokens"] = q_tokens

        if i % 200 == 0 or i == len(query_groups):
            elapsed = time.monotonic() - score_started
            print(
                f"{channel}: {i:,}/{len(query_groups):,} queries | "
                f"{elapsed/60:.1f} min elapsed"
            )

    del query_groups
    gc.collect()
    torch.cuda.empty_cache()

print("Scoring complete.")


In [ ]:
COLBERT_FEATURES = []

for channel in ["full", "text", "title"]:
    score_col = f"colbert_{channel}_maxsim"
    token_col = f"colbert_{channel}_query_tokens"

    if score_col not in all_raw.columns:
        continue

    rank_col = f"colbert_{channel}_rank"
    gap_col = f"colbert_{channel}_gap_to_best"
    mean_col = f"colbert_{channel}_meanmaxsim"

    all_raw[rank_col] = (
        all_raw.groupby(["_split", "query_id"], sort=False)[score_col]
        .rank(method="average", pct=True)
        .astype(np.float32)
    )
    all_raw[gap_col] = (
        all_raw[score_col]
        - all_raw.groupby(["_split", "query_id"], sort=False)[score_col].transform("max")
    ).astype(np.float32)
    all_raw[mean_col] = (
        all_raw[score_col] / all_raw[token_col].clip(lower=1)
    ).astype(np.float32)

    COLBERT_FEATURES.extend([score_col, mean_col, rank_col, gap_col])

assert all_raw[COLBERT_FEATURES].notna().all().all()
assert np.isfinite(all_raw[COLBERT_FEATURES].to_numpy(dtype=np.float64)).all()

train_colbert = all_raw[all_raw["_split"] == "train"][["id"] + COLBERT_FEATURES].copy()
test_colbert = all_raw[all_raw["_split"] == "test"][["id"] + COLBERT_FEATURES].copy()

train_colbert = train_fe[["id"]].merge(train_colbert, on="id", how="left", validate="one_to_one")
test_colbert = test_fe[["id"]].merge(test_colbert, on="id", how="left", validate="one_to_one")

assert train_colbert[COLBERT_FEATURES].notna().all().all()
assert test_colbert[COLBERT_FEATURES].notna().all().all()

train_colbert.to_csv(OUTPUT_DIR / "exp18_colbert_features_train.csv", index=False)
test_colbert.to_csv(OUTPUT_DIR / "exp18_colbert_features_test.csv", index=False)

print("New features:", len(COLBERT_FEATURES))
display(train_colbert[COLBERT_FEATURES].describe().T.round(5))


In [ ]:
del model, transformer_module, tokenizer, query_embeddings, doc_embeddings, query_backing, doc_backing
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("ColBERT features:", len(COLBERT_FEATURES))

In [ ]:
class UnionFind:
    def __init__(self):
        self.parent = {}
        self.size = {}

    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
            self.size[x] = 1
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, a, b):
        pa, pb = self.find(a), self.find(b)
        if pa == pb:
            return
        if self.size[pa] < self.size[pb]:
            pa, pb = pb, pa
        self.parent[pb] = pa
        self.size[pa] += self.size[pb]

uf = UnionFind()
for q, d in zip(train_fe["query_id"].astype(str), train_fe["document_id"].astype(str)):
    uf.union("q:" + q, "d:" + d)

roots = [uf.find("q:" + q) for q in train_fe["query_id"].astype(str)]
component_ids = pd.factorize(np.asarray(roots, dtype=object), sort=False)[0]
labels = train_fe["relevance"].to_numpy(dtype=np.int32)

splits = list(
    GroupKFold(n_splits=N_SPLITS).split(
        np.arange(len(train_fe)), labels, groups=component_ids
    )
)

fold_id = np.full(len(train_fe), -1, dtype=np.int8)
fold_rows = []

for fold, (tr_idx, va_idx) in enumerate(splits, 1):
    tr = train_fe.iloc[tr_idx]
    va = train_fe.iloc[va_idx]

    q_overlap = len(set(tr["query_id"].astype(str)) & set(va["query_id"].astype(str)))
    d_overlap = len(set(tr["document_id"].astype(str)) & set(va["document_id"].astype(str)))
    assert q_overlap == 0 and d_overlap == 0

    fold_id[va_idx] = fold
    fold_rows.append({
        "fold": fold,
        "train_rows": len(tr_idx),
        "valid_rows": len(va_idx),
        "valid_queries": va["query_id"].nunique(),
        "query_overlap": q_overlap,
        "document_overlap": d_overlap,
    })

assert (fold_id > 0).all()
fold_report = pd.DataFrame(fold_rows)
display(fold_report)
fold_report.to_csv(OUTPUT_DIR / "exp18_cv_integrity.csv", index=False)
print("Connected components:", len(np.unique(component_ids)))


In [ ]:
base_oof_scores = CHAMPION13_OOF.astype(np.float64)
base_test_scores = CHAMPION13_TEST.astype(np.float64)
assert len(base_oof_scores) == len(train_fe)
assert len(base_test_scores) == len(test_fe)
assert np.isfinite(base_oof_scores).all() and np.isfinite(base_test_scores).all()
print("Experiment 13.4 Champion OOF:", round(ndcg10(train_fe, base_oof_scores), 6))

In [ ]:
def ndcg10(frame, raw_scores):
    tmp = pd.DataFrame({
        "q": frame["query_id"].astype(str).to_numpy(),
        "y": frame["relevance"].to_numpy(dtype=np.float64),
        "s": np.asarray(raw_scores, dtype=np.float64),
    })
    values = []
    for _, group in tmp.groupby("q", sort=False):
        y = group["y"].to_numpy()
        pred = np.nan_to_num(group["s"].to_numpy(), nan=-1e30, posinf=1e30, neginf=-1e30)
        order = np.argsort(-pred, kind="mergesort")[:10]
        ideal = np.sort(y)[::-1][:10]
        discounts = np.log2(np.arange(2, 2 + len(order)))
        ideal_discounts = np.log2(np.arange(2, 2 + len(ideal)))
        dcg = np.sum((2 ** y[order] - 1) / discounts)
        idcg = np.sum((2 ** ideal - 1) / ideal_discounts)
        values.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(values))

def query_rank(frame, scores):
    values = pd.Series(np.asarray(scores, dtype=np.float64), index=np.arange(len(frame)))
    groups = frame["query_id"].astype(str).reset_index(drop=True)
    return values.groupby(groups).rank(method="average", pct=True).to_numpy(dtype=np.float32)

CAT_COLS = [c for c in ["ingest_source", "query_question_word"] if c in RICH100_FEATURES]
JINA_MISSING_COLS = {"jina_v35_reranker_score", "jina_v35_query_rank"}

FEATURES_EXP18 = RICH100_FEATURES + COLBERT_FEATURES
assert len(FEATURES_EXP18) == len(set(FEATURES_EXP18))

MTRAIN = train_fe[["id"] + RICH100_FEATURES].merge(
    train_colbert, on="id", how="left", validate="one_to_one"
)[FEATURES_EXP18].copy()

MTEST = test_fe[["id"] + RICH100_FEATURES].merge(
    test_colbert, on="id", how="left", validate="one_to_one"
)[FEATURES_EXP18].copy()

for col in FEATURES_EXP18:
    if col in CAT_COLS:
        a = MTRAIN[col].fillna("__MISSING__").astype(str)
        b = MTEST[col].fillna("__MISSING__").astype(str)
        categories = pd.Index(pd.concat([a, b], ignore_index=True).unique())
        MTRAIN[col] = pd.Categorical(a, categories=categories)
        MTEST[col] = pd.Categorical(b, categories=categories)
    else:
        MTRAIN[col] = pd.to_numeric(MTRAIN[col], errors="coerce").replace(
            [np.inf, -np.inf], np.nan
        ).astype(np.float32)
        MTEST[col] = pd.to_numeric(MTEST[col], errors="coerce").replace(
            [np.inf, -np.inf], np.nan
        ).astype(np.float32)

def query_sort(indices):
    indices = np.asarray(indices)
    order = np.argsort(
        train_fe.iloc[indices]["query_id"].astype(str).to_numpy(),
        kind="mergesort",
    )
    sorted_indices = indices[order]
    group_sizes = train_fe.iloc[sorted_indices].groupby("query_id", sort=False).size().tolist()
    return sorted_indices, group_sizes

def fill_fold_matrices(train_indices, valid_indices, features):
    X_train = MTRAIN.iloc[train_indices][features].copy()
    X_valid = MTRAIN.iloc[valid_indices][features].copy()

    for col in features:
        if col in CAT_COLS or col in JINA_MISSING_COLS:
            continue
        median = float(X_train[col].median()) if X_train[col].notna().any() else 0.0
        if not np.isfinite(median):
            median = 0.0
        X_train[col] = X_train[col].fillna(median).astype(np.float32)
        X_valid[col] = X_valid[col].fillna(median).astype(np.float32)

    return X_train, X_valid

def make_ranker(seed, n_estimators=1500):
    return lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        n_estimators=int(n_estimators),
        learning_rate=0.035,
        num_leaves=19,
        min_child_samples=35,
        max_depth=-1,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=8.0,
        label_gain=[0, 1],
        random_state=int(seed),
        n_jobs=-1,
        verbosity=-1,
    )

OOF_EXP18 = np.full(len(train_fe), np.nan, dtype=np.float32)
BEST_ITERATIONS = []
CV_ROWS = []

for fold, (tr_idx, va_idx) in enumerate(splits, 1):
    tr_sorted, tr_groups = query_sort(tr_idx)
    va_sorted, va_groups = query_sort(va_idx)

    X_tr, X_va = fill_fold_matrices(tr_sorted, va_sorted, FEATURES_EXP18)
    model_lgb = make_ranker(RANDOM_STATE + fold)

    model_lgb.fit(
        X_tr,
        labels[tr_sorted],
        group=tr_groups,
        eval_set=[(X_va, labels[va_sorted])],
        eval_group=[va_groups],
        eval_at=[10],
        categorical_feature=[c for c in CAT_COLS if c in FEATURES_EXP18],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    best_iteration = int(model_lgb.best_iteration_ or 1500)
    pred = np.asarray(
        model_lgb.predict(X_va, num_iteration=best_iteration),
        dtype=np.float32,
    )
    OOF_EXP18[va_sorted] = pred
    score = ndcg10(train_fe.iloc[va_sorted], pred)

    BEST_ITERATIONS.append(best_iteration)
    CV_ROWS.append({
        "fold": fold,
        "ndcg10_exp18": score,
        "best_iteration": best_iteration,
        "valid_rows": len(va_sorted),
        "valid_queries": train_fe.iloc[va_sorted]["query_id"].nunique(),
    })

    print(
        f"Fold {fold}/{N_SPLITS}: NDCG@10={score:.6f} "
        f"| best_iteration={best_iteration}"
    )

    del model_lgb, X_tr, X_va, pred
    gc.collect()

assert np.isfinite(OOF_EXP18).all()
cv_results = pd.DataFrame(CV_ROWS)
display(cv_results.round(6))
cv_results.to_csv(OUTPUT_DIR / "exp18_cv_folds.csv", index=False)
np.save(OUTPUT_DIR / "exp18_oof_rich100_colbert.npy", OOF_EXP18)


In [ ]:
base_oof_ndcg = ndcg10(train_fe, base_oof_scores)
exp18_oof_ndcg = ndcg10(train_fe, OOF_EXP18)
print("Experiment 13.4 OOF:", round(base_oof_ndcg, 6))
print("Experiment 18 OOF:", round(exp18_oof_ndcg, 6))

In [ ]:
base_submission = sample_submission.copy()

full_order = np.argsort(
    train_fe["query_id"].astype(str).to_numpy(),
    kind="mergesort",
)
full_group_sizes = train_fe.iloc[full_order].groupby("query_id", sort=False).size().tolist()

X_full = MTRAIN.iloc[full_order][FEATURES_EXP18].copy()
X_test = MTEST[FEATURES_EXP18].copy()

for col in FEATURES_EXP18:
    if col in CAT_COLS or col in JINA_MISSING_COLS:
        continue
    median = float(X_full[col].median()) if X_full[col].notna().any() else 0.0
    if not np.isfinite(median):
        median = 0.0
    X_full[col] = X_full[col].fillna(median).astype(np.float32)
    X_test[col] = X_test[col].fillna(median).astype(np.float32)

final_iterations = max(60, int(np.median(BEST_ITERATIONS)))
final_model = make_ranker(RANDOM_STATE, n_estimators=final_iterations)
final_model.fit(
    X_full,
    labels[full_order],
    group=full_group_sizes,
    categorical_feature=[c for c in CAT_COLS if c in FEATURES_EXP18],
)

test_raw_pred = np.asarray(final_model.predict(X_test), dtype=np.float32)
assert np.isfinite(test_raw_pred).all()

test_rank = query_rank(test_fe, test_raw_pred)
print("Experiment 18 test predictions ready:", len(test_rank))


In [ ]:
CHAMPION18_OOF = OOF_EXP18.copy()
CHAMPION18_TEST = test_rank.copy()
assert np.isfinite(CHAMPION18_OOF).all() and np.isfinite(CHAMPION18_TEST).all()
print("Champion Experiment 18 OOF NDCG@10:", round(ndcg10(train_fe, CHAMPION18_OOF), 6))

for name in [
    "all_raw", "query_table", "query_text_by_key", "query_key_by_id", "doc_text_by_key", "variants",
    "query_embeddings", "doc_embeddings", "query_backing", "doc_backing", "raw_train", "raw_test",
    "train_fe", "test_fe", "train_colbert", "test_colbert", "MTRAIN", "MTEST", "X_full", "X_test",
    "test_raw_pred", "final_model", "OOF_EXP18", "OOF", "FINAL_PRED", "CHAMPION13_OOF", "CHAMPION13_TEST",
    "base_oof_scores", "base_test_scores", "RICH100_FEATURES", "FEATURES_EXP18", "COLBERT_FEATURES",
    "labels", "fold_id", "component_ids", "fold_report", "fold_audit", "MTRAIN", "MTEST",
]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

## 6. Jina v2 и LoRA

In [ ]:
import inspect
from torch.utils.data import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

SEED = 42
MODEL_ID_V2 = "jinaai/jina-reranker-v2-base-multilingual"
MAX_LENGTH = 512
NUM_EPOCHS = 1
PAIR_LIMIT_PER_QUERY = 20
PAIR_BUDGET_PER_FOLD = 6000
TRAIN_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 38
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["mixer.out_proj", "fc1", "fc2"]

if not torch.cuda.is_available():
    raise RuntimeError("Для обучения Jina v2 нужен GPU.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_V2, trust_remote_code=True)

def load_base_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID_V2,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        use_flash_attn=False,
    )
    model.config.use_cache = False
    return model

def score_frame(model, frame, batch_size=16):
    model.eval()
    queries = frame["query"].fillna("").astype(str).tolist()
    docs = (frame["title"].fillna("").astype(str) + "\n" + frame["text"].fillna("").astype(str)).tolist()
    scores = []
    for start in range(0, len(frame), batch_size):
        encoded = tokenizer(
            queries[start:start + batch_size],
            docs[start:start + batch_size],
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {key: value.to(model.device) for key, value in encoded.items()}
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
            logits = model(**encoded).logits.reshape(-1).float()
        scores.extend(logits.cpu().numpy().tolist())
    result = np.asarray(scores, dtype=np.float32)
    if len(result) != len(frame) or not np.isfinite(result).all():
        raise FloatingPointError("Jina scoring вернул неверную длину или NaN/Inf.")
    return result

def build_pair_indices(frame, seed):
    rng = np.random.default_rng(seed)
    labels_local = frame["relevance"].to_numpy(dtype=np.int64)
    pairs = []
    for _, positions in frame.groupby("query_id", sort=False).indices.items():
        idx = np.asarray(positions, dtype=np.int64)
        positive = idx[labels_local[idx] == 1]
        negative = idx[labels_local[idx] == 0]
        if not len(positive) or not len(negative):
            continue
        total = len(positive) * len(negative)
        if total <= PAIR_LIMIT_PER_QUERY:
            local = [(int(p), int(n)) for p in positive for n in negative]
        else:
            chosen = rng.choice(total, size=PAIR_LIMIT_PER_QUERY, replace=False)
            local = [(int(positive[k // len(negative)]), int(negative[k % len(negative)])) for k in chosen]
        pairs.extend(local)
    if not pairs:
        raise ValueError("Не удалось сформировать positive-negative пары.")
    rng.shuffle(pairs)
    if len(pairs) > PAIR_BUDGET_PER_FOLD:
        selected = rng.choice(len(pairs), size=PAIR_BUDGET_PER_FOLD, replace=False)
        pairs = [pairs[i] for i in selected]
    result = np.asarray(pairs, dtype=np.int64)
    if len(result) != PAIR_BUDGET_PER_FOLD:
        raise ValueError(f"Ожидалось {PAIR_BUDGET_PER_FOLD} пар, получено {len(result)}.")
    return result

class PairwiseDataset(Dataset):
    def __init__(self, frame, pair_indices):
        self.frame = frame.reset_index(drop=True)
        self.pairs = np.asarray(pair_indices, dtype=np.int64)
        self.queries = self.frame["query"].to_numpy()
        self.documents = (self.frame["title"].fillna("").astype(str) + "\n" + self.frame["text"].fillna("").astype(str)).to_numpy()

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        pos_idx, neg_idx = self.pairs[index]
        query = self.queries[pos_idx]
        positive = tokenizer(query, self.documents[pos_idx], truncation=True, max_length=MAX_LENGTH)
        negative = tokenizer(query, self.documents[neg_idx], truncation=True, max_length=MAX_LENGTH)
        return {"pos": positive, "neg": negative}

class PairwiseCollator:
    def __init__(self):
        self.pad = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

    def __call__(self, features):
        positive = self.pad([item["pos"] for item in features])
        negative = self.pad([item["neg"] for item in features])
        return {
            **{"pos_" + key: value for key, value in positive.items()},
            **{"neg_" + key: value for key, value in negative.items()},
        }

class PairwiseTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        pos_inputs = {key[4:]: value for key, value in inputs.items() if key.startswith("pos_")}
        neg_inputs = {key[4:]: value for key, value in inputs.items() if key.startswith("neg_")}
        pos_scores = model(**pos_inputs).logits.reshape(-1).float()
        neg_scores = model(**neg_inputs).logits.reshape(-1).float()
        loss = torch.nn.functional.softplus(-(pos_scores - neg_scores)).mean()
        return (loss, {"pos_scores": pos_scores, "neg_scores": neg_scores}) if return_outputs else loss

def make_lora_model():
    model = load_base_model()
    config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        modules_to_save=["classifier"],
        bias="none",
    )
    return get_peft_model(model, config).to("cuda")

def make_training_args(fold_dir, fold_seed):
    kwargs = {
        "output_dir": str(fold_dir),
        "num_train_epochs": NUM_EPOCHS,
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_steps": WARMUP_STEPS,
        "logging_strategy": "no",
        "save_strategy": "no",
        "fp16": True,
        "bf16": False,
        "optim": "adamw_torch",
        "lr_scheduler_type": "linear",
        "max_grad_norm": 1.0,
        "report_to": "none",
        "remove_unused_columns": False,
        "dataloader_num_workers": 0,
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "seed": fold_seed,
        "data_seed": fold_seed,
    }
    supported = inspect.signature(TrainingArguments.__init__).parameters
    return TrainingArguments(**{key: value for key, value in kwargs.items() if key in supported})

folds = splits
oof_zero = np.full(len(train), np.nan, dtype=np.float32)
oof_lora = np.full(len(train), np.nan, dtype=np.float32)
test_zero_by_fold = []
test_lora_by_fold = []
fold_metrics = []

for fold_id, (train_idx, valid_idx) in enumerate(folds, 1):
    print(f"\nFold {fold_id}/{N_SPLITS}")
    train_fold = train.iloc[train_idx].reset_index(drop=True)
    valid_fold = train.iloc[valid_idx].reset_index(drop=True)
    pair_indices = build_pair_indices(train_fold, SEED + fold_id)

    base_model = load_base_model().to("cuda")
    zero_valid = score_frame(base_model, valid_fold)
    zero_test = score_frame(base_model, test)
    del base_model
    gc.collect()
    torch.cuda.empty_cache()

    model = make_lora_model()
    dataset = PairwiseDataset(train_fold, pair_indices)
    trainer = PairwiseTrainer(
        model=model,
        args=make_training_args(OUTPUT_DIR / f"exp20_fold_{fold_id}", SEED + fold_id),
        train_dataset=dataset,
        data_collator=PairwiseCollator(),
    )
    trainer.train()

    lora_valid = score_frame(trainer.model, valid_fold, batch_size=8)
    lora_test = score_frame(trainer.model, test, batch_size=8)

    oof_zero[valid_idx] = zero_valid
    oof_lora[valid_idx] = lora_valid
    test_zero_by_fold.append(zero_test)
    test_lora_by_fold.append(lora_test)
    fold_metrics.append({
        "fold": fold_id,
        "train_rows": len(train_idx),
        "valid_rows": len(valid_idx),
        "pairs": len(pair_indices),
        "zero_shot_ndcg10": ndcg10(train.iloc[valid_idx], zero_valid),
        "lora_ndcg10": ndcg10(train.iloc[valid_idx], lora_valid),
    })

    del trainer, model, dataset, train_fold, valid_fold
    gc.collect()
    torch.cuda.empty_cache()

assert np.isfinite(oof_zero).all() and np.isfinite(oof_lora).all()
test_zero = np.mean(np.vstack(test_zero_by_fold), axis=0).astype(np.float32)
test_lora = np.mean(np.vstack(test_lora_by_fold), axis=0).astype(np.float32)
fold_results = pd.DataFrame(fold_metrics)
print("Experiment 20 zero-shot OOF:", round(ndcg10(train, oof_zero), 6))
print("Experiment 20 LoRA OOF:", round(ndcg10(train, oof_lora), 6))
display(fold_results.round(6))


## 7. подбор веса ансамбля

In [ ]:
exp18_oof_rank = query_rank(train, CHAMPION18_OOF)
lora_oof_rank = query_rank(train, oof_lora)
exp18_test_rank = query_rank(test, CHAMPION18_TEST)
lora_test_rank = query_rank(test, test_lora)

blend_rows = []
for lora_weight in np.linspace(0.0, 1.0, 101):
    blend = (1.0 - lora_weight) * exp18_oof_rank + lora_weight * lora_oof_rank
    blend_rows.append({
        "lora_weight": float(lora_weight),
        "exp18_weight": float(1.0 - lora_weight),
        "oof_ndcg10": ndcg10(train, blend),
    })

blend_results = pd.DataFrame(blend_rows).sort_values("oof_ndcg10", ascending=False).reset_index(drop=True)
best = blend_results.iloc[0]
FINAL_W_LORA = float(best["lora_weight"])
FINAL_W_EXP18 = float(best["exp18_weight"])

final_oof = FINAL_W_EXP18 * exp18_oof_rank + FINAL_W_LORA * lora_oof_rank
final_test = FINAL_W_EXP18 * exp18_test_rank + FINAL_W_LORA * lora_test_rank

print("Experiment 18 weight:", round(FINAL_W_EXP18, 2))
print("Jina v2 LoRA weight:", round(FINAL_W_LORA, 2))
print("Final OOF NDCG@10:", round(ndcg10(train, final_oof), 6))


## 8. Submission

In [ ]:
test_scores_by_id = pd.Series(
    final_test,
    index=test["id"].astype(str),
    dtype=np.float64,
)
submission = pd.DataFrame({"id": sample_submission["id"].astype(str)})
submission["prediction"] = submission["id"].map(test_scores_by_id)

if submission["prediction"].isna().any():
    missing = submission.loc[submission["prediction"].isna(), "id"].head(10).tolist()
    raise ValueError(f"Для некоторых ID нет prediction: {missing}")
if len(submission) != len(test) or not submission["id"].is_unique:
    raise AssertionError("Не совпадает количество строк или есть повторяющиеся ID.")
if not np.isfinite(submission["prediction"].to_numpy(dtype=np.float64)).all():
    raise FloatingPointError("В submission есть NaN/Inf.")

SUBMISSION_PATH = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path.cwd() / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
print("Готово:", SUBMISSION_PATH)
print("Строк:", len(submission), "| Колонки:", list(submission.columns))
display(submission.head())
